# Types of Machine Learning Embeddings (TensorFlow, OpenCV, TensorFlow Text)

This notebook provides a practical taxonomy of embedding types with:
- Concepts and subtypes
- Purpose for each subtype
- Code examples with comments
- Do and Don't checklists
- Tips and tricks
- Summary and application use cases

## Scope
Focused on workflows built with **TensorFlow**, **tensorflow-text**, and **OpenCV**.

## Learning Path For First-Time Learners

Start here and move forward in order:
1. Setup and basic text embeddings
2. Image embeddings and structured/tabular embeddings
3. Multi-modal embeddings and architecture-based embeddings
4. Specialised families: word prediction, neural, matrix factorization, graph, recurrent, probabilistic
5. Transformer-based embeddings and mini applications

If you are new, run one section at a time and inspect the printed shapes before moving forward.

### What To Focus On First
- Understand what one embedding vector represents
- Compare dense vectors by cosine similarity
- Learn how padding, masking, and normalization change results
- Keep each example small before scaling up

### Suggested Run Order and Checkpoints

For a first pass, run the notebook in this order:
1. Setup
2. A) Modality-based embeddings
3. B) Architecture-based embeddings
4. C) Per-subtype examples
5. D) Image and video deep dive
6. E) Mini application use cases
7. F) Transformer-based embeddings
8. G) Additional embedding families

Recommended checkpoints:
- After A: confirm text, image, structured, and multimodal shapes
- After B: confirm every architecture cell prints a valid embedding shape
- After D: confirm OpenCV-based cells behave on missing features and small inputs
- After E: confirm every application example runs end-to-end
- After G: confirm all family cells print expected latent dimensions

If a cell fails, fix the smallest section first and rerun only that section before moving on.

## 0) Setup

In [15]:
# Core imports
import os
import numpy as np
import tensorflow as tf

# Optional imports for this notebook
try:
    import tensorflow_text as tf_text
    TF_TEXT_AVAILABLE = True
except Exception:
    TF_TEXT_AVAILABLE = False

try:
    import cv2
    OPENCV_AVAILABLE = True
except Exception:
    OPENCV_AVAILABLE = False

print('TensorFlow:', tf.__version__)
print('tensorflow-text available:', TF_TEXT_AVAILABLE)
print('OpenCV available:', OPENCV_AVAILABLE)

TensorFlow: 2.22.0-dev0+selfbuilt
tensorflow-text available: True
OpenCV available: True


## 0.5) tensorflow-text vs Keras Preprocessing: Replacement Guide

Use this section when you want to replace a Keras text preprocessing method with a tensorflow-text equivalent, or the other way around.

| Task | Keras approach | tensorflow-text approach | Notes |
|---|---|---|---|
| Tokenize text | `TextVectorization` | `WhitespaceTokenizer`, `UnicodeScriptTokenizer`, custom split ops | tensorflow-text gives more control over token boundaries |
| Convert tokens to ids | `TextVectorization` or `StringLookup` | `StaticVocabularyTable` or lookup tables | both can map strings to ids |
| Subword / WordPiece | limited built-in support | `SentencepieceTokenizer`, WordPiece-style tokenization | better for multilingual and rare words |
| Sequence padding | `output_sequence_length` | `pad_sequences` or `to_tensor` | both can create fixed-length sequences |
| Detokenize / inspect tokens | limited | `detokenize`, `UnicodeScriptTokenizer` outputs | useful for debugging and alignment |

Rule: use Keras preprocessing when the pipeline is simple and you want fast integration; use tensorflow-text when tokenization or subword logic must match production NLP behavior exactly.

## A) Grouped by Data Type (Modality)

Below is a modality-first view of embedding subtypes and their purpose.

### A1. Text Embeddings

| Subtype | Purpose | Typical Tools |
|---|---|---|
| Token embeddings | Map discrete tokens to dense vectors for neural models | TensorFlow Embedding layer |
| Subword embeddings | Handle OOV and morphology using pieces (WordPiece/BPE-like) | tensorflow-text tokenizers + Embedding |
| Character embeddings | Robustness to typos/noise and morphology | TF strings + char lookup |
| Sentence embeddings | Encode sentence-level semantics for similarity/retrieval | Pooling over token states / encoders |
| Contextual embeddings | Dynamic token meaning based on context | Transformer encoders |

In [17]:
# Example: token/subword style pipeline with TensorFlow + tensorflow-text style ops
texts = tf.constant([
    'machine learning improves search',
    'deep learning powers modern NLP'
])

# Basic whitespace split (works without tensorflow-text)
tokens = tf.strings.split(texts)
flat_tokens = tokens.flat_values

# Build a simple vocab from observed tokens for demonstration
unique_tokens, _ = tf.unique(flat_tokens)
vocab = tf.concat([tf.constant(['<PAD>', '<UNK>']), unique_tokens], axis=0)

# Map tokens -> ids
table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(vocab, tf.range(tf.shape(vocab)[0], dtype=tf.int64)),
    num_oov_buckets=1
)
token_ids = table.lookup(flat_tokens)

# Token embedding layer
embedding_dim = 16
emb_layer = tf.keras.layers.Embedding(input_dim=int(vocab.shape[0]) + 1, output_dim=embedding_dim)
embedded_tokens = emb_layer(token_ids)

print('Vocab size:', int(vocab.shape[0]))
print('Token IDs shape:', token_ids.shape)
print('Embedded shape:', embedded_tokens.shape)

Vocab size: 10
Token IDs shape: (9,)
Embedded shape: (9, 16)


**Do (Text Embeddings):**
- Normalize text consistently (case, punctuation, unicode strategy).
- Reserve OOV and PAD ids explicitly.
- Tune embedding dimension to data scale (small data -> smaller dims).
- Use subwords when domain has misspellings or many rare words.

**Don't (Text Embeddings):**
- Do not mix tokenization methods between training and serving.
- Do not assume larger dimension always improves generalization.
- Do not ignore sequence length truncation effects in production.

**Tips and Tricks:**
- Start with 64 to 256 dims for medium-size corpora; shrink if overfitting.
- Use average pooling baseline before complex attention heads.
- Monitor nearest neighbors qualitatively for embedding health checks.

**Summary:** Text embeddings map language units to vectors that capture semantics and syntax at different granularities.

**Application Use Cases:** semantic search, intent classification, duplicate question detection, recommendation ranking.

### A2. Image Embeddings

| Subtype | Purpose | Typical Tools |
|---|---|---|
| CNN feature embeddings | Compact semantic image features for classification/retrieval | TensorFlow CNN backbones |
| Patch embeddings | Represent local patches (e.g., ViT style) | TensorFlow Conv/Dense patch projectors |
| Keypoint descriptor embeddings | Match local regions across images | OpenCV SIFT/ORB descriptors |
| Region embeddings | Encode object/ROI level features | OpenCV ROI + TF encoder |

In [18]:
# Example: global image embedding with a tiny CNN in TensorFlow
images = tf.random.uniform(shape=(4, 64, 64, 3), minval=0.0, maxval=1.0)

cnn_embedder = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, activation='relu'),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64)  # final image embedding size
])

image_embeddings = cnn_embedder(images)
print('Image embedding shape:', image_embeddings.shape)

# Optional OpenCV local descriptor example
if OPENCV_AVAILABLE:
    # Create a synthetic grayscale image for feature extraction demo
    img = (np.random.rand(128, 128) * 255).astype(np.uint8)

    if hasattr(cv2, 'ORB_create'):
        orb = cv2.ORB_create()
        keypoints, descriptors = orb.detectAndCompute(img, None)
        desc_shape = None if descriptors is None else descriptors.shape
        print('ORB keypoints:', len(keypoints), '| descriptor shape:', desc_shape)
    else:
        print('ORB not available in this OpenCV build.')

Image embedding shape: (4, 64)
ORB keypoints: 267 | descriptor shape: (267, 32)


**Do (Image Embeddings):**
- Keep preprocessing identical for training/inference (resize, color conversion, normalization).
- Use data augmentation for robust embeddings.
- L2-normalize embeddings for cosine-based retrieval systems.

**Don't (Image Embeddings):**
- Do not compare embeddings from differently preprocessed pipelines.
- Do not evaluate only top-1 accuracy if your goal is retrieval quality.

**Tips and Tricks:**
- Start from pretrained backbones and fine-tune.
- For low-latency search, use smaller dimensions with metric-learning loss.
- In OpenCV pipelines, ORB is faster while SIFT is often more robust.

**Summary:** Image embeddings transform pixels or local descriptors into vectors suitable for semantic matching and downstream vision tasks.

**Application Use Cases:** visual similarity search, product matching, face/person retrieval, industrial defect lookup.

### A3. Structured Data Embeddings (Categorical, Numerical, Time)

| Subtype | Purpose | Typical Tools |
|---|---|---|
| Categorical embeddings | Replace one-hot with dense vectors for high-cardinality features | TensorFlow Embedding layer |
| Numerical projection embeddings | Project normalized continuous features to latent space | Dense projection layers |
| Time-series embeddings | Represent windows/temporal patterns compactly | 1D CNN/RNN/Transformer in TF |

In [19]:
# Example: categorical + numerical embeddings for tabular ML
batch_size = 5

# Two categorical features (e.g., user_id_bucket, country_id)
cat1 = tf.constant([1, 3, 2, 5, 4], dtype=tf.int32)
cat2 = tf.constant([2, 2, 1, 3, 4], dtype=tf.int32)

# Numerical features
num = tf.random.normal((batch_size, 3))

cat_emb1 = tf.keras.layers.Embedding(input_dim=100, output_dim=8)(cat1)
cat_emb2 = tf.keras.layers.Embedding(input_dim=20, output_dim=4)(cat2)
num_proj = tf.keras.layers.Dense(8, activation='relu')(num)

# Concatenate into one unified embedding vector
final_tabular_embedding = tf.concat([cat_emb1, cat_emb2, num_proj], axis=-1)
print('Tabular embedding shape:', final_tabular_embedding.shape)

Tabular embedding shape: (5, 20)


**Do (Structured Embeddings):**
- Handle unseen category ids with OOV buckets.
- Normalize numerical features before projection.
- For time series, align windowing and forecasting horizon correctly.

**Don't (Structured Embeddings):**
- Do not leak future information in temporal features.
- Do not use huge embedding dimensions for low-cardinality features.

**Tips and Tricks:**
- Rule-of-thumb for categorical dim: min(50, round(cardinality^0.25)).
- Share embeddings for semantically related categorical fields when valid.

**Summary:** Structured embeddings let mixed feature types live in one latent space, improving model capacity and memory efficiency.

**Application Use Cases:** recommender systems, churn prediction, fraud detection, demand forecasting.

### A4. Multi-Modal Embeddings

| Subtype | Purpose | Typical Tools |
|---|---|---|
| Text-Image joint embeddings | Place text and images in one semantic space | TF dual-encoders + OpenCV preprocessing |
| Sensor-image-text fusion embeddings | Unified representation from multiple data channels | TF feature fusion networks |

In [20]:
# Example: simple dual-encoder style fusion (toy)
text_vec = tf.random.normal((4, 64))   # pretend this came from a text encoder
img_vec = tf.random.normal((4, 64))    # pretend this came from an image encoder

# Project both to the same space and normalize for cosine similarity
proj = tf.keras.layers.Dense(32)
text_proj = tf.math.l2_normalize(proj(text_vec), axis=-1)
img_proj = tf.math.l2_normalize(proj(img_vec), axis=-1)

# Similarity matrix between text and images
similarity = tf.linalg.matmul(text_proj, img_proj, transpose_b=True)
print('Similarity matrix shape:', similarity.shape)

Similarity matrix shape: (4, 4)


**Do (Multi-Modal Embeddings):**
- Align training pairs carefully (text-image pairing quality matters).
- Use temperature scaling in contrastive objectives.
- Evaluate both retrieval directions (text->image and image->text).

**Don't (Multi-Modal Embeddings):**
- Do not rely only on one modality at inference if both are expected.
- Do not ignore modality-specific normalization/preprocessing.

**Tips and Tricks:**
- Hard-negative mining usually improves retrieval quality.
- Batch size strongly impacts contrastive learning stability.

**Summary:** Multi-modal embeddings create a shared semantic geometry across heterogeneous inputs.

**Application Use Cases:** text-image search, visual question retrieval, product catalog matching with image and description.

## B) Grouped by Model Architecture

Architecture-first view of embeddings and their purpose.

| Architecture Subtype | Purpose | Typical Libraries |
|---|---|---|
| Lookup-table embeddings | Learn vector per discrete id | TensorFlow Embedding |
| CNN embeddings | Learn local-to-global visual/text patterns | TensorFlow Keras CNN |
| RNN/LSTM embeddings | Sequence-order-aware representations | TensorFlow Keras RNN |
| Transformer embeddings | Contextual, long-range dependency encoding | TensorFlow Transformer blocks, tensorflow-text tokenization |
| Autoencoder embeddings | Compressed latent representation via reconstruction | TensorFlow encoder-decoder |
| Metric-learning embeddings | Geometry optimized by distance constraints | TF custom training (triplet/contrastive losses) |
| Graph embeddings (conceptual) | Node/edge semantics in latent space | TF-GNN style ecosystems |
| Hybrid/fusion embeddings | Combine multiple encoders/modalities | TensorFlow + OpenCV preprocessing |

### B2. Separate Code Cell Per Architecture Type

Each architecture embedding subtype below has its own standalone code cell with inline comments and purpose.

In [25]:
# Lookup-table Embedding
# Purpose: map discrete IDs (tokens/categories) into trainable dense vectors.

x_ids = tf.constant([[1, 2, 3], [3, 4, 0]], dtype=tf.int32)
lookup_emb = tf.keras.layers.Embedding(input_dim=100, output_dim=16)(x_ids)
lookup_pooled = tf.reduce_mean(lookup_emb, axis=1)  # sequence -> fixed-size embedding
print('lookup_pooled shape:', lookup_pooled.shape)

lookup_pooled shape: (2, 16)


In [26]:
# CNN Embedding
# Purpose: learn local patterns and compress them into a semantic vector.

cnn_out = tf.keras.Sequential([
    tf.keras.layers.Conv1D(32, 3, activation='relu'),
    tf.keras.layers.GlobalMaxPool1D(),
    tf.keras.layers.Dense(16)
])(lookup_emb)
print('cnn_out shape:', cnn_out.shape)

cnn_out shape: (2, 16)


In [27]:
# Autoencoder Embedding
# Purpose: compress features into a latent vector that reconstructs inputs.

dense_in = tf.random.normal((2, 20))
latent = tf.keras.layers.Dense(8, activation='relu')(dense_in)
recon = tf.keras.layers.Dense(20)(latent)
print('latent shape:', latent.shape)
print('recon shape:', recon.shape)

latent shape: (2, 8)
recon shape: (2, 20)


In [28]:
# Transformer Embedding
# Purpose: produce contextual embeddings using self-attention across the sequence.

attn = tf.keras.layers.MultiHeadAttention(num_heads=2, key_dim=8)(lookup_emb, lookup_emb)
tfm_out = tf.keras.layers.GlobalAveragePooling1D()(attn)
print('tfm_out shape:', tfm_out.shape)

tfm_out shape: (2, 16)


In [29]:
# RNN/LSTM Embedding
# Purpose: encode order-sensitive sequential context into a dense representation.

rnn_out = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(12))(lookup_emb)
print('rnn_out shape:', rnn_out.shape)

rnn_out shape: (2, 24)


In [30]:
# Hybrid/Fusion Embedding
# Purpose: fuse embeddings from different encoders/modalities into one joint vector.

text_emb = tf.random.normal((3, 16))
image_emb = tf.random.normal((3, 16))

fusion_in = tf.concat([text_emb, image_emb], axis=-1)
fusion_emb = tf.keras.layers.Dense(20, activation='relu')(fusion_in)
print('fusion_emb shape:', fusion_emb.shape)

fusion_emb shape: (3, 20)


In [31]:
# Graph Embedding (Conceptual GNN-style)
# Purpose: combine node features with graph structure to get node embeddings.

num_nodes = 5
node_feat = tf.random.normal((num_nodes, 8))
adj = tf.constant([
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1],
], dtype=tf.float32)

# One message-passing step: aggregate neighbor features, then project
agg = tf.matmul(adj, node_feat)
node_emb = tf.keras.layers.Dense(8, activation='relu')(agg)
print('node_emb shape:', node_emb.shape)

node_emb shape: (5, 8)


In [32]:
# Metric-Learning Embedding
# Purpose: shape embedding geometry so similar pairs are close and dissimilar pairs are far.

a = tf.random.normal((4, 16))
p = a + 0.05 * tf.random.normal((4, 16))      # positive samples near anchors
n = tf.random.normal((4, 16))                  # negatives sampled independently

# Triplet loss (toy): max(0, d(a,p) - d(a,n) + margin)
margin = 0.2
d_ap = tf.norm(a - p, axis=-1)
d_an = tf.norm(a - n, axis=-1)
triplet_loss = tf.reduce_mean(tf.nn.relu(d_ap - d_an + margin))
print('triplet_loss:', float(triplet_loss))

triplet_loss: 0.0


### Do and Don't by Architecture

**Lookup-table embeddings**
- Do: regularize (L2/dropout) and reserve OOV/PAD ids.
- Don't: oversize dimensions for tiny vocabularies.

**CNN embeddings**
- Do: tune kernel sizes for local pattern scales.
- Don't: ignore stride/padding effects on detail loss.

**RNN/LSTM embeddings**
- Do: use masking and consider bidirectional encoders.
- Don't: expect best long-context performance vs transformers.

**Transformer embeddings**
- Do: use proper positional encoding and attention masks.
- Don't: train from scratch on tiny datasets without transfer learning.

**Autoencoder embeddings**
- Do: validate latent space utility for downstream task, not only reconstruction loss.
- Don't: assume perfect recon implies best discriminative embedding.

**Metric-learning embeddings**
- Do: use hard/semi-hard negatives carefully.
- Don't: evaluate with classification accuracy alone; use retrieval metrics too.

**Hybrid/fusion embeddings**
- Do: calibrate each encoder scale before fusion.
- Don't: concatenate many noisy features without selection.

### Tips and Tricks (Global)

1. Normalize vectors when distance-based retrieval is the objective.
2. Keep a simple baseline embedding model before adding complexity.
3. Watch for train/serve skew in tokenization and image preprocessing.
4. Use ANN indexes (FAISS/ScaNN style) for large-scale embedding retrieval.
5. Track drift by monitoring nearest-neighbor quality over time.
6. Evaluate embeddings with both intrinsic and task metrics:
   - Intrinsic: cosine neighborhood quality, clustering silhouette
   - Task: recall@k, MRR, NDCG, downstream F1/AUC

## Final Summary

- **Modality grouping** helps decide *what unit* should be embedded (text token, image region, category id, multimodal pair).
- **Architecture grouping** helps decide *how* embeddings are learned (lookup, CNN, RNN, transformer, autoencoder, metric learning, fusion).
- In production, the highest impact comes from:
  - strict preprocessing consistency,
  - correct objective/loss for the use case,
  - robust evaluation beyond a single metric,
  - and stable serving-time vector normalization + indexing.

## Mini Exercises (Lab Ready)

These short exercises are designed to be runnable and easy to modify.

What you get:
- Exercise 1: Text embedding retrieval (cosine similarity)
- Exercise 2: Image embedding retrieval with TensorFlow CNN
- Exercise 3: Tabular/user-item style embedding similarity

Tip: Run in order from Exercise 1 to Exercise 3.

In [22]:
# Exercise 1: Text embedding retrieval (toy)
# Goal: see which query sentence is most similar to each candidate sentence.

import tensorflow as tf

sentences = tf.constant([
    "machine learning improves ranking",
    "deep learning for computer vision",
    "tokenization and embeddings for nlp",
    "image retrieval with feature vectors",
])

query = tf.constant(["embeddings for text search"])

# Build simple token ids
all_text = tf.concat([sentences, query], axis=0)
tokens = tf.strings.split(tf.strings.lower(all_text))
flat = tokens.flat_values
uniq, _ = tf.unique(flat)
vocab = tf.concat([tf.constant(["<PAD>", "<UNK>"]), uniq], axis=0)

vocab_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(vocab, tf.range(tf.shape(vocab)[0], dtype=tf.int64)),
    num_oov_buckets=1,
)

# Convert each sentence to mean pooled token embedding
embed_dim = 24
emb = tf.keras.layers.Embedding(input_dim=4096, output_dim=embed_dim)

def sentence_embed(text_batch):
    rag = tf.strings.split(tf.strings.lower(text_batch))
    dense = rag.to_tensor(default_value="<PAD>")
    dense_ids = vocab_table.lookup(dense)
    vecs = emb(dense_ids)

    # Mask out PAD tokens before averaging
    mask = tf.cast(tf.not_equal(dense, "<PAD>"), tf.float32)
    mask = tf.expand_dims(mask, axis=-1)
    summed = tf.reduce_sum(vecs * mask, axis=1)
    counts = tf.reduce_sum(mask, axis=1) + 1e-8
    return summed / counts

sent_vecs = tf.math.l2_normalize(sentence_embed(sentences), axis=-1)
query_vec = tf.math.l2_normalize(sentence_embed(query), axis=-1)

# Cosine similarity (query vs each sentence)
scores = tf.squeeze(tf.matmul(query_vec, sent_vecs, transpose_b=True), axis=0)
best_idx = tf.argmax(scores).numpy()

print("Query:", query.numpy()[0].decode())
for i, s in enumerate(sentences.numpy()):
    print(f"  cand[{i}] score={float(scores[i]):.4f} :: {s.decode()}")
print("Best match index:", best_idx)
print("Best match text:", sentences.numpy()[best_idx].decode())

Query: embeddings for text search
  cand[0] score=0.2683 :: machine learning improves ranking
  cand[1] score=0.1304 :: deep learning for computer vision
  cand[2] score=0.4021 :: tokenization and embeddings for nlp
  cand[3] score=-0.0555 :: image retrieval with feature vectors
Best match index: 2
Best match text: tokenization and embeddings for nlp


In [23]:
# Exercise 2: Image embedding retrieval (toy)
# Goal: embed a query image and find nearest image in a small gallery.

import tensorflow as tf

# Synthetic image gallery
gallery = tf.random.uniform((6, 64, 64, 3), 0.0, 1.0)
query_img = tf.random.uniform((1, 64, 64, 3), 0.0, 1.0)

# Small CNN encoder
img_encoder = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(48),
])

gallery_vec = tf.math.l2_normalize(img_encoder(gallery), axis=-1)
query_vec = tf.math.l2_normalize(img_encoder(query_img), axis=-1)

sim = tf.squeeze(tf.matmul(query_vec, gallery_vec, transpose_b=True), axis=0)
nearest = tf.argmax(sim).numpy()

print("Similarity scores:", [round(float(v), 4) for v in sim.numpy()])
print("Nearest gallery index:", nearest)

# Optional OpenCV descriptor check (if available)
try:
    import cv2
    import numpy as np
    gray = (gallery[nearest].numpy().mean(axis=-1) * 255).astype("uint8")
    if hasattr(cv2, "ORB_create"):
        orb = cv2.ORB_create()
        kps, desc = orb.detectAndCompute(gray, None)
        print("OpenCV ORB keypoints on nearest image:", len(kps))
        print("ORB descriptor shape:", None if desc is None else desc.shape)
except Exception as e:
    print("OpenCV descriptor block skipped:", str(e)[:120])

Similarity scores: [0.9999, 0.9999, 0.9999, 0.9999, 1.0, 0.9998]
Nearest gallery index: 4
OpenCV ORB keypoints on nearest image: 0
ORB descriptor shape: None


In [24]:
# Exercise 3: Tabular embedding similarity (user-item style)
# Goal: combine categorical + numeric features into one embedding and compare users.

import tensorflow as tf

# Toy user batch (user_id bucket, country_id, numeric activity features)
user_id = tf.constant([1, 2, 3, 4, 5], dtype=tf.int32)
country_id = tf.constant([2, 1, 2, 3, 1], dtype=tf.int32)
activity = tf.constant([
    [0.2, 1.5, 0.1],
    [0.4, 1.1, 0.3],
    [1.2, 0.1, 0.7],
    [1.0, 0.2, 0.8],
    [0.3, 1.4, 0.2],
], dtype=tf.float32)

user_emb = tf.keras.layers.Embedding(input_dim=100, output_dim=8)(user_id)
country_emb = tf.keras.layers.Embedding(input_dim=10, output_dim=4)(country_id)
num_proj = tf.keras.layers.Dense(8, activation="relu")(activity)

combined = tf.concat([user_emb, country_emb, num_proj], axis=-1)
combined = tf.math.l2_normalize(combined, axis=-1)

# Pairwise similarity matrix among users
sim_matrix = tf.matmul(combined, combined, transpose_b=True)
print("User-user similarity matrix shape:", sim_matrix.shape)
print(tf.round(sim_matrix * 1000) / 1000)

# Find nearest neighbor for user 0 (excluding self)
scores_u0 = sim_matrix[0]
scores_u0 = tf.tensor_scatter_nd_update(scores_u0, indices=[[0]], updates=[-1.0])
nn_idx = tf.argmax(scores_u0).numpy()
print("Nearest neighbor for user[0] is user[{}]".format(nn_idx))

User-user similarity matrix shape: (5, 5)
tf.Tensor(
[[1.    0.983 0.402 0.574 0.994]
 [0.983 1.    0.449 0.638 0.991]
 [0.402 0.449 1.    0.931 0.42 ]
 [0.574 0.638 0.931 1.    0.598]
 [0.994 0.991 0.42  0.598 1.   ]], shape=(5, 5), dtype=float32)
Nearest neighbor for user[0] is user[4]


## C) Separate Code Cell Per Modality Subtype

Each subtype below is isolated in its own code cell and includes comments explaining purpose and behavior.

In [33]:
# C1) Text Token Embedding
# Purpose: map token IDs to dense vectors for NLP models.

tok_ids = tf.constant([[1, 2, 3], [2, 4, 0]], dtype=tf.int32)
tok_vec = tf.keras.layers.Embedding(input_dim=50, output_dim=8)(tok_ids)
print('token embedding shape:', tok_vec.shape)

token embedding shape: (2, 3, 8)


In [34]:
# C2) Text Subword Embedding
# Purpose: represent subword pieces to handle rare/OOV terms.

sub_tokens = tf.constant([[b'mach', b'ine', b'learn'], [b'deep', b'learn', b'ing']])
sub_flat = tf.reshape(sub_tokens, [-1])
sub_vocab, _ = tf.unique(sub_flat)
sub_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(sub_vocab, tf.range(tf.shape(sub_vocab)[0], dtype=tf.int64)),
    num_oov_buckets=1,
)
sub_ids = sub_table.lookup(sub_tokens)
sub_vec = tf.keras.layers.Embedding(input_dim=200, output_dim=8)(sub_ids)
print('subword embedding shape:', sub_vec.shape)

subword embedding shape: (2, 3, 8)


In [35]:
# C3) Text Character Embedding
# Purpose: robustly encode words at character level.

chars = tf.constant([[b'm', b'l', b'\x00'], [b'n', b'l', b'p']])
char_flat = tf.reshape(chars, [-1])
char_vocab, _ = tf.unique(char_flat)
char_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(char_vocab, tf.range(tf.shape(char_vocab)[0], dtype=tf.int64)),
    num_oov_buckets=1,
)
char_ids = char_table.lookup(chars)
char_vec = tf.keras.layers.Embedding(input_dim=128, output_dim=6)(char_ids)
print('character embedding shape:', char_vec.shape)

character embedding shape: (2, 3, 6)


In [36]:
# C4) Sentence Embedding
# Purpose: generate one vector for a full sentence (good for retrieval/classification).

sent_ids = tf.constant([[1, 4, 3, 0], [2, 2, 5, 6]], dtype=tf.int32)
sent_tok = tf.keras.layers.Embedding(input_dim=100, output_dim=10)(sent_ids)
sent_vec = tf.reduce_mean(sent_tok, axis=1)
print('sentence embedding shape:', sent_vec.shape)

sentence embedding shape: (2, 10)


In [37]:
# C5) Contextual Text Embedding (Transformer-style)
# Purpose: token vectors depend on surrounding tokens via self-attention.

ctx_ids = tf.constant([[1, 2, 3, 4], [4, 3, 2, 1]], dtype=tf.int32)
ctx_tok = tf.keras.layers.Embedding(input_dim=100, output_dim=12)(ctx_ids)
ctx_attn = tf.keras.layers.MultiHeadAttention(num_heads=2, key_dim=6)(ctx_tok, ctx_tok)
ctx_vec = tf.keras.layers.GlobalAveragePooling1D()(ctx_attn)
print('contextual sentence embedding shape:', ctx_vec.shape)

contextual sentence embedding shape: (2, 12)


In [38]:
# C6) Image CNN Feature Embedding
# Purpose: encode full image semantics into a compact vector.

img_batch = tf.random.uniform((2, 64, 64, 3))
img_vec = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, activation='relu'),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(24),
])(img_batch)
print('cnn image embedding shape:', img_vec.shape)

cnn image embedding shape: (2, 24)


In [39]:
# C7) Image Patch Embedding
# Purpose: represent local image patches as vectors (ViT-style building block).

img = tf.random.uniform((1, 32, 32, 3))
# Conv with stride=patch size acts as patch projector
patch_proj = tf.keras.layers.Conv2D(filters=16, kernel_size=8, strides=8)(img)
# Flatten spatial grid to sequence of patch vectors
patch_seq = tf.reshape(patch_proj, (1, -1, 16))
print('patch sequence shape:', patch_seq.shape)

patch sequence shape: (1, 16, 16)


In [40]:
# C8) Image Keypoint Descriptor Embedding (OpenCV ORB)
# Purpose: encode local keypoints for matching/retrieval.

if OPENCV_AVAILABLE:
    rnd = (np.random.rand(128, 128) * 255).astype(np.uint8)
    orb_local = cv2.ORB_create()
    kp, des = orb_local.detectAndCompute(rnd, None)
    print('num keypoints:', len(kp))
    print('descriptor shape:', None if des is None else des.shape)
else:
    print('OpenCV not available')

num keypoints: 270
descriptor shape: (270, 32)


In [41]:
# C9) Image Region Embedding (ROI)
# Purpose: embed a selected region-of-interest rather than full image.

full_img = tf.random.uniform((1, 64, 64, 3))
roi = full_img[:, 16:48, 16:48, :]  # center crop as ROI
roi_vec = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(12),
])(roi)
print('roi embedding shape:', roi_vec.shape)

roi embedding shape: (1, 12)


In [42]:
# C10) Structured Categorical Embedding
# Purpose: replace sparse one-hot categorical features with dense vectors.

cat_ids = tf.constant([1, 5, 2, 7], dtype=tf.int32)
cat_vec = tf.keras.layers.Embedding(input_dim=100, output_dim=8)(cat_ids)
print('categorical embedding shape:', cat_vec.shape)

categorical embedding shape: (4, 8)


In [43]:
# C11) Structured Numerical Projection Embedding
# Purpose: project normalized continuous features into latent space.

num_feat = tf.random.normal((4, 5))
num_vec = tf.keras.layers.Dense(8, activation='relu')(num_feat)
print('numerical projection shape:', num_vec.shape)

numerical projection shape: (4, 8)


In [44]:
# C12) Time-Series Embedding
# Purpose: encode temporal windows into fixed-size vectors.

ts = tf.random.normal((3, 20, 4))  # batch, time, features
ts_vec = tf.keras.Sequential([
    tf.keras.layers.Conv1D(16, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(10),
])(ts)
print('time-series embedding shape:', ts_vec.shape)

time-series embedding shape: (3, 10)


In [45]:
# C13) Multi-Modal Text-Image Joint Embedding
# Purpose: place text and image into the same vector space.

t_vec = tf.random.normal((3, 16))
i_vec = tf.random.normal((3, 16))
proj_layer = tf.keras.layers.Dense(12)
t_joint = tf.math.l2_normalize(proj_layer(t_vec), axis=-1)
i_joint = tf.math.l2_normalize(proj_layer(i_vec), axis=-1)
sim_joint = tf.matmul(t_joint, i_joint, transpose_b=True)
print('joint similarity matrix shape:', sim_joint.shape)

joint similarity matrix shape: (3, 3)


In [46]:
# C14) Multi-Modal Sensor-Image-Text Fusion Embedding
# Purpose: fuse multiple modality vectors into one representation.

sensor_vec = tf.random.normal((3, 6))
text_vec = tf.random.normal((3, 10))
image_vec = tf.random.normal((3, 12))

fusion_input = tf.concat([sensor_vec, text_vec, image_vec], axis=-1)
fusion_out = tf.keras.layers.Dense(16, activation='relu')(fusion_input)
print('sensor-image-text fusion shape:', fusion_out.shape)

sensor-image-text fusion shape: (3, 16)


## D) Image & Video Embedding Types — One Cell Per Subtype

Each cell below is fully standalone (imports included).

### Image Embedding Subtypes
| Subtype | What it encodes | Key use case |
|---|---|---|
| CNN Global Feature | Full image semantics | Classification, retrieval |
| HOG Descriptor | Edge/gradient structure | Pedestrian detection, OCR |
| Color Histogram | Pixel color distribution | Image deduplication, scene matching |
| ORB / BoW Visual Word | Local keypoint occurrences | Document/logo matching |
| Spatial Pyramid Pooling | Spatial layout + semantics | Fine-grained recognition |
| Attention-Weighted Image | Salient region emphasis | VQA, captioning, e-commerce |

### Video Embedding Subtypes
| Subtype | What it encodes | Key use case |
|---|---|---|
| Frame-Level (CNN per frame) | Static appearance per frame | Scene classification |
| Temporal (3D CNN) | Short-range spatio-temporal patterns | Action recognition |
| LSTM over CNN Frames | Long-range temporal dependencies | Activity/event classification |
| Optical Flow (OpenCV + TF) | Motion between consecutive frames | Motion classification, anomaly |
| Frame Difference | Coarse motion signal | Change detection, surveillance |

In [47]:
# IMAGE EMBEDDING TYPE 1: CNN Global Feature Embedding
# -------------------------------------------------------
# Purpose : Summarise full-image semantics into one dense vector using
#           stacked convolutions + global pooling.
# Use case: Image retrieval, product similarity search, face verification.

import tensorflow as tf

# Simulate a small batch of RGB images (4 images, 64x64)
images = tf.random.uniform(shape=(4, 64, 64, 3), minval=0.0, maxval=1.0)

# Build a compact CNN encoder
cnn_encoder = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling2D(),   # collapses (H, W) -> one vector
    tf.keras.layers.Dense(64),                  # final embedding size
])

embeddings = cnn_encoder(images)
# L2-normalise so cosine similarity == dot product
embeddings = tf.math.l2_normalize(embeddings, axis=-1)
print("CNN global embedding shape:", embeddings.shape)   # (4, 64)
# Pairwise cosine similarity matrix (useful for retrieval evaluation)
sim = tf.matmul(embeddings, embeddings, transpose_b=True)
print("Similarity matrix shape:", sim.shape)             # (4, 4)

CNN global embedding shape: (4, 64)
Similarity matrix shape: (4, 4)


In [48]:
# IMAGE EMBEDDING TYPE 2: HOG (Histogram of Oriented Gradients) Descriptor
# -------------------------------------------------------------------------
# Purpose : Encode local edge/gradient structure of an image as a histogram
#           vector. Captures shape without colour.
# Use case: Pedestrian detection, OCR pre-processing, object detection.

import cv2
import numpy as np
import tensorflow as tf

def hog_embed(images_np: np.ndarray, cell=(8, 8), block=(2, 2), nbins=9) -> np.ndarray:
    """Compute HOG descriptor for each grayscale image in the batch.
    Returns an array of shape (batch, descriptor_dim)."""
    win_size = (images_np.shape[2], images_np.shape[1])  # (W, H)
    hog = cv2.HOGDescriptor(
        win_size,
        (block[0] * cell[0], block[1] * cell[1]),  # blockSize
        (cell[0], cell[1]),                          # blockStride (one cell)
        (cell[0], cell[1]),                          # cellSize
        nbins,
    )
    results = []
    for img in images_np:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
        desc = hog.compute(gray)
        results.append(desc.flatten())
    return np.array(results, dtype=np.float32)

# Create synthetic RGB images (4 images, 64x64)
imgs_np = (np.random.rand(4, 64, 64, 3) * 255).astype(np.uint8)
hog_vecs = hog_embed(imgs_np)
print("HOG descriptor shape:", hog_vecs.shape)   # (4, descriptor_dim)

# Optionally project to smaller embedding for a neural model
hog_t = tf.constant(hog_vecs)
hog_emb = tf.keras.layers.Dense(32, activation='relu')(hog_t)
print("HOG projected embedding:", hog_emb.shape)  # (4, 32)

HOG descriptor shape: (4, 1764)
HOG projected embedding: (4, 32)


In [49]:
# IMAGE EMBEDDING TYPE 3: Color Histogram Embedding
# --------------------------------------------------
# Purpose : Represent pixel colour distribution as a histogram vector.
#           Fast, lightweight, no training needed.
# Use case: Image deduplication, scene-type matching, thumbnail comparison.

import cv2
import numpy as np
import tensorflow as tf

def color_hist_embed(images_np: np.ndarray, bins: int = 32) -> np.ndarray:
    """Compute a per-channel colour histogram and concatenate them."""
    results = []
    for img in images_np:
        channels = []
        for ch in range(img.shape[-1]):  # iterate R, G, B channels
            hist = cv2.calcHist([img], [ch], None, [bins], [0, 256])
            hist = cv2.normalize(hist, hist).flatten()
            channels.append(hist)
        results.append(np.concatenate(channels))
    return np.array(results, dtype=np.float32)

imgs_np = (np.random.rand(4, 64, 64, 3) * 255).astype(np.uint8)
hist_vecs = color_hist_embed(imgs_np, bins=32)
print("Color histogram shape:", hist_vecs.shape)  # (4, 96) = 3 x 32

# Optionally project to a smaller latent vector
hist_t = tf.constant(hist_vecs)
hist_emb = tf.keras.layers.Dense(24, activation='relu')(hist_t)
print("Color hist projected embedding:", hist_emb.shape)

Color histogram shape: (4, 96)
Color hist projected embedding: (4, 24)


In [50]:
# IMAGE EMBEDDING TYPE 4: ORB Bag-of-Visual-Words Embedding
# ----------------------------------------------------------
# Purpose : Encode the frequency of visual "words" (quantised local keypoint
#           descriptors) as a fixed-length histogram. Translation-invariant.
# Use case: Logo/document retrieval, near-duplicate detection.

import cv2
import numpy as np
import tensorflow as tf

def orb_bow_embed(images_np: np.ndarray, vocab_size: int = 64) -> np.ndarray:
    """ORB descriptors -> mini bag-of-words via random codebook.
    In production replace the random codebook with a k-means codebook."""
    orb = cv2.ORB_create(nfeatures=200)
    rng = np.random.default_rng(42)

    # Build a random codebook (stand-in for a real k-means codebook)
    codebook = rng.standard_normal((vocab_size, 32)).astype(np.float32)

    results = []
    for img in images_np:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, des = orb.detectAndCompute(gray, None)
        bow = np.zeros(vocab_size, dtype=np.float32)
        if des is not None:
            des_f = des.astype(np.float32)
            # Assign each descriptor to the nearest codebook word
            dists = np.linalg.norm(
                des_f[:, None, :] - codebook[None, :, :], axis=-1
            )  # (n_kp, vocab_size)
            words = np.argmin(dists, axis=-1)
            for w in words:
                bow[w] += 1
            bow /= (bow.sum() + 1e-8)  # normalise to frequency
        results.append(bow)
    return np.array(results, dtype=np.float32)

imgs_np = (np.random.rand(4, 64, 64, 3) * 255).astype(np.uint8)
bow_vecs = orb_bow_embed(imgs_np)
print("BoW embedding shape:", bow_vecs.shape)      # (4, 64)

# Project to a smaller latent space
bow_t = tf.constant(bow_vecs)
bow_emb = tf.keras.layers.Dense(16, activation='relu')(bow_t)
print("BoW projected embedding:", bow_emb.shape)

BoW embedding shape: (4, 64)
BoW projected embedding: (4, 16)


In [51]:
# IMAGE EMBEDDING TYPE 5: Spatial Pyramid Pooling (SPP) Embedding
# ----------------------------------------------------------------
# Purpose : Pool features at multiple spatial scales so the embedding
#           captures both global layout and fine-grained local structure.
# Use case: Fine-grained recognition (car model, bird species), multi-scale retrieval.

import tensorflow as tf

images = tf.random.uniform(shape=(3, 64, 64, 3))

# Shared CNN backbone — produces a feature map
backbone = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
])
feat_map = backbone(images)  # (3, 32, 32, 64)

# SPP: pool at 3 scales (1x1, 2x2, 4x4) and concatenate
def spp_pool(feat, levels=(1, 2, 4)):
    parts = []
    for l in levels:
        # AdaptiveAvgPool approximation via strided average pooling
        pooled = tf.keras.layers.AveragePooling2D(
            pool_size=(feat.shape[1] // l, feat.shape[2] // l),
            strides=(feat.shape[1] // l, feat.shape[2] // l),
        )(feat)
        parts.append(tf.reshape(pooled, (tf.shape(pooled)[0], -1)))
    return tf.concat(parts, axis=-1)

spp_vec = spp_pool(feat_map)
print("SPP vector shape:", spp_vec.shape)     # (3, 64*(1+4+16))=1344

# Final projection to fixed embedding size
spp_emb = tf.keras.layers.Dense(64)(spp_vec)
print("SPP embedding shape:", spp_emb.shape)  # (3, 64)

SPP vector shape: (3, 1344)
SPP embedding shape: (3, 64)


In [52]:
# IMAGE EMBEDDING TYPE 6: Attention-Weighted Image Embedding
# -----------------------------------------------------------
# Purpose : Apply spatial self-attention to a CNN feature map so the
#           embedding emphasises salient regions rather than averaging uniformly.
# Use case: Visual question answering, product attribute focus, captioning support.

import tensorflow as tf

images = tf.random.uniform(shape=(3, 64, 64, 3))

# CNN backbone producing spatial feature map
backbone = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
])
feat_map = backbone(images)                         # (3, 32, 32, 32)
B, H, W, C = tf.shape(feat_map)[0], 32, 32, 32

# Flatten spatial dims -> sequence of (H*W) position vectors
seq = tf.reshape(feat_map, (B, H * W, C))          # (3, 1024, 32)

# Attention score per spatial position
attn_scores = tf.keras.layers.Dense(1)(seq)         # (3, 1024, 1)
attn_weights = tf.nn.softmax(attn_scores, axis=1)   # normalise over positions

# Weighted sum: focus on attended regions
attended = tf.reduce_sum(seq * attn_weights, axis=1)  # (3, 32)

# Final projection
attn_emb = tf.keras.layers.Dense(24)(attended)
print("Attention-weighted embedding shape:", attn_emb.shape)  # (3, 24)

Attention-weighted embedding shape: (3, 24)


### Video Embedding Subtypes

In [53]:
# VIDEO EMBEDDING TYPE 1: Frame-Level CNN Embedding + Mean Pooling
# -----------------------------------------------------------------
# Purpose : Extract a CNN feature per video frame, then average across time
#           to produce a single clip embedding. Simple and fast.
# Use case: Scene-type classification, short clip retrieval.

import tensorflow as tf

# Simulate a mini-batch of 2 video clips, each with 8 frames of 32x32 RGB
# In production these would be actual decoded video frames.
videos = tf.random.uniform(shape=(2, 8, 32, 32, 3))

# A shared CNN frame encoder — weights are shared across all time steps
frame_encoder = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.GlobalAveragePooling2D(),   # (H, W, C) -> (C,) per frame
    tf.keras.layers.Dense(64, activation='relu'),
])

# Apply the frame encoder independently to each of the 8 frames
# TimeDistributed wraps any layer to run it per time-step
td_encoder = tf.keras.layers.TimeDistributed(frame_encoder)
frame_embs = td_encoder(videos)               # (2, 8, 64)

# Temporal mean pooling: collapse 8 frame embeddings -> 1 clip embedding
clip_emb = tf.reduce_mean(frame_embs, axis=1)  # (2, 64)
clip_emb = tf.math.l2_normalize(clip_emb, axis=-1)
print("Frame-level CNN clip embedding shape:", clip_emb.shape)  # (2, 64)

Frame-level CNN clip embedding shape: (2, 64)


In [54]:
# VIDEO EMBEDDING TYPE 2: 3D CNN (C3D-style) Spatio-Temporal Embedding
# ----------------------------------------------------------------------
# Purpose : Apply 3-D convolutions across (T, H, W) to jointly model
#           spatial appearance and short-range temporal patterns in one pass.
# Use case: Action recognition, gesture detection, sport highlight detection.

import tensorflow as tf

# Simulate a mini-batch: 2 clips, 8 frames, 32x32 RGB
clips = tf.random.uniform(shape=(2, 8, 32, 32, 3))

# 3-D convolution: kernel spans time (t), height (h), width (w)
c3d_net = tf.keras.Sequential([
    # First 3D conv block — captures short-range motion (e.g. 3-frame window)
    tf.keras.layers.Conv3D(16, kernel_size=(3, 3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPool3D(pool_size=(1, 2, 2)),   # only halve spatial, keep time
    # Second 3D conv block — larger receptive field over time
    tf.keras.layers.Conv3D(32, kernel_size=(3, 3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPool3D(pool_size=(2, 2, 2)),   # halve both spatial and temporal
    # Collapse all dimensions to a fixed embedding
    tf.keras.layers.GlobalAveragePooling3D(),
    tf.keras.layers.Dense(64, activation='relu'),     # (batch, 64)
])

clip_emb = c3d_net(clips)
print("3D-CNN clip embedding shape:", clip_emb.shape)  # (2, 64)

3D-CNN clip embedding shape: (2, 64)


In [55]:
# VIDEO EMBEDDING TYPE 3: LSTM over CNN Frame Embeddings
# -------------------------------------------------------
# Purpose : Run a recurrent (LSTM) layer over the sequence of per-frame CNN
#           embeddings so the model can track long-range temporal dependencies.
# Use case: Activity/event classification, video-level sentiment, sport analysis.

import tensorflow as tf

# Simulate 2 clips, 12 frames each, 32x32 RGB
videos = tf.random.uniform(shape=(2, 12, 32, 32, 3))

# Step 1: shared CNN encodes each frame to a 64-d vector
frame_cnn = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation='relu'),
])
# TimeDistributed applies the CNN to every time-step independently
frame_seq = tf.keras.layers.TimeDistributed(frame_cnn)(videos)  # (2, 12, 64)

# Step 2: LSTM reads the frame sequence and summarises it
# return_sequences=False => only the final hidden state is output
lstm_emb = tf.keras.layers.LSTM(48, return_sequences=False)(frame_seq)  # (2, 48)
# Bidirectional LSTM is also common — it reads forward and backward:
# lstm_emb = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(48))(frame_seq)

lstm_emb = tf.math.l2_normalize(lstm_emb, axis=-1)
print("LSTM video embedding shape:", lstm_emb.shape)  # (2, 48)

LSTM video embedding shape: (2, 48)


In [56]:
# VIDEO EMBEDDING TYPE 4: Optical Flow Embedding (OpenCV + TF Dense Projection)
# -------------------------------------------------------------------------------
# Purpose : Compute dense per-pixel motion vectors (optical flow) between
#           consecutive frames, then project them into a latent embedding.
#           Captures how pixels move — pure motion, no appearance bias.
# Use case: Action recognition (tennis swing vs. golf swing), anomaly detection,
#           robotics visual odometry.

import cv2
import numpy as np
import tensorflow as tf

def compute_flow_batch(frames_np: np.ndarray) -> np.ndarray:
    """Compute Farneback dense optical flow between consecutive frames.
    frames_np: (T, H, W, 3) uint8 for a single clip.
    Returns mean flow magnitude per consecutive pair, shape (T-1, H*W).
    """
    flows = []
    for t in range(len(frames_np) - 1):
        prev = cv2.cvtColor(frames_np[t], cv2.COLOR_RGB2GRAY)
        nxt  = cv2.cvtColor(frames_np[t + 1], cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(
            prev, nxt, None,
            pyr_scale=0.5, levels=3, winsize=15, iterations=3,
            poly_n=5, poly_sigma=1.2, flags=0,
        )  # (H, W, 2): dx, dy per pixel
        mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2).flatten()
        flows.append(mag)
    return np.array(flows, dtype=np.float32)  # (T-1, H*W)

# Simulate 4 video clips with 5 frames each
T, H, W = 5, 32, 32
clips_np = [(np.random.rand(T, H, W, 3) * 255).astype(np.uint8) for _ in range(4)]

all_flow_vecs = []
for clip in clips_np:
    flow_seq = compute_flow_batch(clip)           # (T-1, H*W)
    # Aggregate: mean over temporal pairs -> (H*W,) per clip
    clip_flow = flow_seq.mean(axis=0)
    all_flow_vecs.append(clip_flow)

flow_batch = tf.constant(np.array(all_flow_vecs))  # (4, H*W)
print("Raw flow vector shape:", flow_batch.shape)    # (4, 1024)

# Project into a compact embedding
flow_emb = tf.keras.layers.Dense(48, activation='relu')(flow_batch)
print("Optical flow embedding shape:", flow_emb.shape)  # (4, 48)

Raw flow vector shape: (4, 1024)
Optical flow embedding shape: (4, 48)


In [57]:
# VIDEO EMBEDDING TYPE 5: Frame Difference (Temporal Gradient) Embedding
# -----------------------------------------------------------------------
# Purpose : Compute pixel-wise absolute difference between consecutive frames.
#           Lightweight motion signal — no optical flow solver required.
# Use case: Change/anomaly detection, surveillance (motion trigger),
#           fast pre-filter before a heavier model.

import numpy as np
import tensorflow as tf

# Simulate 4 clips, 6 frames each, 32x32 greyscale
T, H, W = 6, 32, 32
clips_np = np.random.rand(4, T, H, W, 1).astype(np.float32)

# Frame difference: |frame[t+1] - frame[t]| for each consecutive pair
diff = np.abs(clips_np[:, 1:, :, :, :] - clips_np[:, :-1, :, :, :])
# diff shape: (4, T-1, H, W, 1)

# Aggregate motion energy: mean over time and space -> (4, 1) per clip
motion_energy = diff.mean(axis=(1, 2, 3, 4), keepdims=True)  # (4, 1)

# Flatten each difference frame to a vector, then pool
diff_flat = diff.reshape(4, T - 1, H * W)               # (4, T-1, H*W)
diff_mean = diff_flat.mean(axis=1)                        # (4, H*W)

diff_t = tf.constant(diff_mean)
# Project to compact embedding
diff_emb = tf.keras.layers.Dense(32, activation='relu')(diff_t)
print("Frame difference embedding shape:", diff_emb.shape)   # (4, 32)
print("Mean motion energy per clip:", motion_energy.squeeze().tolist())

Frame difference embedding shape: (4, 32)
Mean motion energy per clip: [0.33256837725639343, 0.33407455682754517, 0.3293681740760803, 0.33458977937698364]


## E) Mini Application Use Cases

Each application uses embeddings as the core building block. Every cell is self-contained with inline comments.

| # | Application | Embedding strategy | Libraries |
|---|---|---|---|
| E1 | QA — answer from document | Sentence embedding + cosine retrieval | TF, Keras |
| E2 | Translation | Seq2Seq encoder-decoder LSTM | TF, Keras |
| E3 | Code generation (Java / TS / Python) | Language-conditioned character LSTM | TF, Keras |
| E4 | Image to text (captioning) | CNN encoder + LSTM decoder + attention | TF, Keras, OpenCV |
| E5 | Image search by text | Dual-encoder (text ↔ image) cosine retrieval | TF, Keras |
| E6 | Image search by image | CNN feature gallery + cosine retrieval | TF, Keras, OpenCV |
| E7 | Text to image | Conditional VAE text → image | TF, Keras |
| E8 | Text to video | CNN frame decoder over time | TF, Keras |
| E9 | Text similarity | Embedding cosine + token overlap (BM25-style) | TF, TF-Text, Keras |
| E10 | NER (Named Entity Recognition) | BiLSTM + token classification | TF, Keras |
| E11 | POS (Part-of-Speech tagging) | BiLSTM + softmax over POS tags | TF, Keras |

### E1) QA — Answer from Document (Extractive)

**Strategy**: embed each sentence with a lookup+pooling encoder, then find the sentence most similar to the question via cosine similarity.  
**Alternatives**: TF-IDF weighted embedding · BM25 + dense re-rank · BiDAF attention reader.

In [58]:
# E1) QA — Extractive Answer Retrieval from a Document
# =====================================================
# Approach : Embed each document sentence + the question using a shared
#            lookup embedding + mean-pool encoder.
#            Return the sentence with the highest cosine similarity.

import tensorflow as tf
import numpy as np

# ----- tiny vocabulary & document ----------------------------------------
VOCAB = ["<PAD>","the","cat","sat","on","mat","dog","ran","fast","through","park",
         "where","is","sleeping","which","animal","fastest","moved"]
word2id = {w: i for i, w in enumerate(VOCAB)}
EMB_DIM = 16

document_sentences = [
    "the cat sat on the mat",
    "the dog ran fast through the park",
    "the cat is sleeping on the mat",
]
question = "which animal is sleeping"

# ----- helper: tokenise + pad a list of sentences -----------------------
def tokenise(sentences, word2id, maxlen=10):
    ids = [[word2id.get(w, 0) for w in s.split()] for s in sentences]
    return tf.keras.preprocessing.sequence.pad_sequences(ids, maxlen=maxlen, padding='post')

doc_ids = tf.constant(tokenise(document_sentences, word2id))   # (3, 10)
q_ids   = tf.constant(tokenise([question], word2id))            # (1, 10)

# ----- shared sentence encoder (lookup + mean pooling) -------------------
emb_layer  = tf.keras.layers.Embedding(len(VOCAB), EMB_DIM, mask_zero=True)
proj_layer = tf.keras.layers.Dense(32)

def encode(ids):
    # Mask padding tokens before mean-pooling
    mask   = tf.cast(ids != 0, tf.float32)[:, :, tf.newaxis]  # (B, T, 1)
    vecs   = emb_layer(ids) * mask                              # (B, T, D)
    pooled = tf.reduce_sum(vecs, axis=1) / (tf.reduce_sum(mask, axis=1) + 1e-8)
    return tf.math.l2_normalize(proj_layer(pooled), axis=-1)

doc_embs = encode(doc_ids)   # (3, 32)
q_emb    = encode(q_ids)     # (1, 32)

# ----- cosine retrieval --------------------------------------------------
scores      = tf.matmul(q_emb, doc_embs, transpose_b=True)  # (1, 3)
best_idx    = int(tf.argmax(scores, axis=-1).numpy()[0])
print("Question  :", question)
print("Best match:", document_sentences[best_idx])
print("Scores    :", scores.numpy())

Question  : which animal is sleeping
Best match: the cat is sleeping on the mat
Scores    : [[0.15565816 0.12638906 0.47004336]]


### E2) Translation (Seq2Seq Encoder–Decoder)

**Strategy**: Encoder LSTM reads source tokens → context vector. Decoder LSTM generates target tokens one step at a time.  
**Alternatives**: Attention-augmented Bahdanau decoder · Transformer (MultiHeadAttention) · subword tokenisation with tensorflow-text.

In [59]:
# E2) Translation — Seq2Seq Encoder-Decoder LSTM
# ================================================
# Architecture:
#   Encoder  : Embedding → LSTM(64)  → context (h, c)
#   Decoder  : Embedding → LSTM(64, initial_state=context) → Dense(vocab) → token
# This cell builds + runs a single forward pass (training forward pass pattern).

import tensorflow as tf
import numpy as np

# ----- toy bilingual vocabulary ------------------------------------------
SRC_VOCAB = ["<PAD>","<SOS>","<EOS>","hello","world","good","morning","how","are","you"]
TGT_VOCAB = ["<PAD>","<SOS>","<EOS>","bonjour","monde","bon","matin","comment","allez","vous"]
SV, TV = len(SRC_VOCAB), len(TGT_VOCAB)
EMB, UNITS = 16, 64

# ----- model components --------------------------------------------------
enc_emb  = tf.keras.layers.Embedding(SV, EMB, mask_zero=True)
enc_lstm = tf.keras.layers.LSTM(UNITS, return_state=True)

dec_emb  = tf.keras.layers.Embedding(TV, EMB, mask_zero=True)
dec_lstm = tf.keras.layers.LSTM(UNITS, return_sequences=True, return_state=True)
dec_out  = tf.keras.layers.Dense(TV, activation='softmax')   # token probability

# ----- synthetic mini-batch: "hello world" → "bonjour monde" -------------
#  src: [hello(3), world(4), <EOS>(2)]
src = tf.constant([[3, 4, 2]])    # (1, 3)
#  tgt_in:  [<SOS>(1), bonjour(3), monde(4)]
#  tgt_out: [bonjour(3), monde(4), <EOS>(2)]
tgt_in  = tf.constant([[1, 3, 4]])
tgt_out = tf.constant([[3, 4, 2]])

# ----- encoder forward pass ----------------------------------------------
enc_seq, enc_h, enc_c = enc_lstm(enc_emb(src))   # context hidden + cell states

# ----- decoder forward pass (teacher forcing) ----------------------------
dec_seq, _, _ = dec_lstm(dec_emb(tgt_in), initial_state=[enc_h, enc_c])
logits = dec_out(dec_seq)   # (1, 3, TV) — prob over TGT_VOCAB at each step

# ----- loss --------------------------------------------------------------
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
loss    = loss_fn(tgt_out, logits)

pred_ids = tf.argmax(logits, axis=-1).numpy()[0]
print("Predicted token ids:", pred_ids)
print("Predicted tokens   :", [TGT_VOCAB[i] for i in pred_ids])
print("Training loss      :", float(loss))

Predicted token ids: [7 1 1]
Predicted tokens   : ['comment', '<SOS>', '<SOS>']
Training loss      : 2.301074981689453


### E3) Code Generation — Java / TypeScript / Python

**Strategy**: Language-conditioned LSTM where the target language (Java/TS/Python) is embedded as a 1-token prefix, giving the decoder a language-specific context vector.  
**Alternatives**: Language ID embedding concatenated to each decoder step · separate decoder heads per language · Transformer with language token.

In [60]:
# E3) Code Generation — Language-Conditioned LSTM
# ================================================
# The encoder receives a natural-language description.
# A language-ID embedding (Java=0 / TypeScript=1 / Python=2) is prepended
# to the decoder input so the same model generates different-language code.

import tensorflow as tf
import numpy as np

# ----- vocabularies (shared; real system would use a code tokeniser) -----
NL_VOCAB   = ["<PAD>","<EOS>","print","hello","world","function","define","class"]
CODE_VOCAB = ["<PAD>","<SOS>","<EOS>",
              # Java tokens
              "System.out.println","(","\"Hello\"",")",";",
              # TypeScript tokens
              "console.log","'Hello'",
              # Python tokens
              "print"]
NV, CV = len(NL_VOCAB), len(CODE_VOCAB)
LANG_VOCAB = 3   # 0=Java, 1=TypeScript, 2=Python
EMB, UNITS = 16, 64

# ----- model layers -------------------------------------------------------
nl_emb   = tf.keras.layers.Embedding(NV, EMB, mask_zero=True)
lang_emb = tf.keras.layers.Embedding(LANG_VOCAB, EMB)  # language conditioning
enc_lstm = tf.keras.layers.LSTM(UNITS, return_state=True)

code_emb = tf.keras.layers.Embedding(CV, EMB, mask_zero=True)
dec_lstm = tf.keras.layers.LSTM(UNITS, return_sequences=True, return_state=True)
dec_proj = tf.keras.layers.Dense(CV, activation='softmax')

def encode_nl(nl_ids):
    _, h, c = enc_lstm(nl_emb(nl_ids))
    return h, c

def decode_one_step(lang_id, code_in_ids, enc_h, enc_c):
    """Decode one step given language ID and input token sequence."""
    lang_vec  = lang_emb(tf.constant([[lang_id]]))             # (1, 1, EMB)
    code_vecs = code_emb(code_in_ids)                          # (1, T, EMB)
    # Prepend language token to decoder sequence
    full_seq  = tf.concat([lang_vec, code_vecs], axis=1)       # (1, T+1, EMB)
    out, _, _ = dec_lstm(full_seq, initial_state=[enc_h, enc_c])
    return dec_proj(out)   # (1, T+1, CV)

# ----- forward pass: "print hello world" in three languages --------------
nl_ids = tf.constant([[2, 3, 4]])   # "print hello world"
enc_h, enc_c = encode_nl(nl_ids)

for lang_name, lang_id in [("Java", 0), ("TypeScript", 1), ("Python", 2)]:
    code_in = tf.constant([[1]])    # <SOS>
    logits  = decode_one_step(lang_id, code_in, enc_h, enc_c)
    token   = int(tf.argmax(logits[0, -1]).numpy())
    print(f"[{lang_name:12s}] first predicted token: {CODE_VOCAB[token]!r}")

[Java        ] first predicted token: 'print'
[TypeScript  ] first predicted token: 'console.log'
[Python      ] first predicted token: 'console.log'


### E4) Image to Text (Captioning)

**Strategy**: CNN encoder (feature map) + LSTM decoder with additive attention (Bahdanau). At each step the decoder attends to different spatial regions.  
**Alternatives**: ViT patch encoder → Transformer decoder · OpenCV preprocessing + CNN · CLIP image encoder + GPT decoder.

In [63]:
# E4) Image to Text — CNN Encoder + Attention LSTM Decoder
# ==========================================================
# Pipeline:
#   1. CNN backbone → spatial feature map  (H', W', C')
#   2. Flatten spatial dims → sequence of region vectors
#   3. Bahdanau attention selects which region to focus on each step
#   4. LSTM decoder generates caption tokens

import tensorflow as tf
import numpy as np

CAP_VOCAB = ["<PAD>","<SOS>","<EOS>","a","cat","dog","sits","on","the","mat"]
CV = len(CAP_VOCAB)
EMB, UNITS = 16, 64

# ----- CNN backbone: image → spatial features ----------------------------
cnn_backbone = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
])

# ----- Bahdanau (additive) attention ------------------------------------
W1 = tf.keras.layers.Dense(UNITS)   # project encoder features
W2 = tf.keras.layers.Dense(UNITS)   # project decoder hidden state
V  = tf.keras.layers.Dense(1)       # score per region

def attention(enc_output, dec_h):
    """enc_output: (B, T, C)  dec_h: (B, UNITS)"""
    dec_h_exp = tf.expand_dims(dec_h, 1)                   # (B, 1, UNITS)
    score     = V(tf.nn.tanh(W1(enc_output) + W2(dec_h_exp)))  # (B, T, 1)
    weights   = tf.nn.softmax(score, axis=1)               # (B, T, 1)
    context   = tf.reduce_sum(weights * enc_output, axis=1)    # (B, C)
    return context, weights

# ----- decoder layers ---------------------------------------------------
cap_emb  = tf.keras.layers.Embedding(CV, EMB, mask_zero=True)
dec_lstm = tf.keras.layers.LSTMCell(UNITS)
dec_proj = tf.keras.layers.Dense(CV, activation='softmax')

# ----- forward pass: 1 image, 3 decode steps ----------------------------
image    = tf.random.uniform((1, 32, 32, 3))
feat_map = cnn_backbone(image)                          # (1, 16, 16, 64)
B, H, W, C_ = 1, 16, 16, 64
enc_seq  = tf.reshape(feat_map, (B, H * W, C_))         # (1, 256, 64)

# Initialise decoder state
h = tf.zeros((B, UNITS))
c = tf.zeros((B, UNITS))

token = 1   # <SOS>
print("Caption token ids: [<SOS>", end="")
for _ in range(3):
    ctx, _ = attention(enc_seq, h)                              # (1, 64)
    emb    = cap_emb(tf.reshape(token, (1, 1)))                 # (1, 1, EMB)
    inp    = tf.concat([emb[:, 0, :], ctx], axis=-1)            # (1, EMB+64)
    out, [h, c] = dec_lstm(inp, states=[h, c])
    logits = dec_proj(out)
    token  = int(tf.argmax(logits, axis=-1).numpy()[0])
    print(",", CAP_VOCAB[token], end="")
print("]")

Caption token ids: [<SOS>, sits, sits, sits]


### E5) Image Search Based on Text (Text → Image Retrieval)

**Strategy**: Dual-encoder — separate text and image towers project into a shared embedding space. At query time cosine-rank all gallery images.  
**Alternatives**: CLIP-style contrastive pre-training · late-fusion re-ranking · hash-based ANN with FAISS.

In [64]:
# E5) Image Search Based on Text — Dual-Encoder (CLIP-style)
# ===========================================================
# Both towers project into the SAME 32-d embedding space.
# Query: text description  →  retrieve the most similar gallery images.

import tensorflow as tf
import numpy as np

SHARED_DIM = 32

# ----- Text tower -------------------------------------------------------
VOCAB = ["<PAD>","cat","dog","car","plane","bird","sitting","running","flying","on","a","mat"]
word2id = {w: i for i, w in enumerate(VOCAB)}

def build_text_encoder():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(len(VOCAB), 16, mask_zero=True),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(SHARED_DIM),
    ])

# ----- Image tower ------------------------------------------------------
def build_image_encoder():
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(SHARED_DIM),
    ])

text_enc  = build_text_encoder()
image_enc = build_image_encoder()

# ----- Synthetic gallery: 6 images with labels --------------------------
gallery_labels = ["cat on mat", "dog running", "bird flying", "car", "plane", "cat sitting"]
gallery_imgs   = tf.random.uniform((6, 32, 32, 3))

# Encode gallery
gallery_emb = tf.math.l2_normalize(image_enc(gallery_imgs), axis=-1)  # (6, 32)

# ----- Text query -------------------------------------------------------
query = "cat sitting"
q_ids = [[word2id.get(w, 0) for w in query.split()]]
q_ids = tf.keras.preprocessing.sequence.pad_sequences(q_ids, maxlen=6)
q_emb = tf.math.l2_normalize(text_enc(tf.constant(q_ids)), axis=-1)    # (1, 32)

# ----- Cosine retrieval -------------------------------------------------
scores   = tf.matmul(q_emb, gallery_emb, transpose_b=True)[0]          # (6,)
top3_idx = tf.argsort(scores, direction='DESCENDING').numpy()[:3]

print("Query:", query)
print("Top-3 matched images:")
for rank, idx in enumerate(top3_idx, 1):
    print(f"  {rank}. '{gallery_labels[idx]}'  score={scores[idx]:.4f}")

Query: cat sitting
Top-3 matched images:
  1. 'dog running'  score=0.1660
  2. 'cat on mat'  score=0.1647
  3. 'bird flying'  score=0.1646


### E6) Image Search Based on Image (Image → Image Retrieval)

**Strategy**: Extract CNN feature embedding for query and gallery images; rank gallery by cosine similarity. OpenCV ORB descriptors serve as an alternative fast baseline.  
**Alternatives**: Siamese network triplet loss · product quantisation · OpenCV `BFMatcher` on ORB descriptors.

In [65]:
# E6) Image Search Based on Image — CNN Feature Gallery + ORB Fallback
# ====================================================================
# Approach A : CNN feature embedding + cosine similarity (deep retrieval)
# Approach B : OpenCV ORB keypoint descriptors + BFMatcher (classical CV)

import tensorflow as tf
import numpy as np
import cv2

# ----- Approach A: CNN embedding gallery --------------------------------
cnn = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(32),
])

gallery_imgs = tf.random.uniform((8, 32, 32, 3))   # 8 gallery images
query_img    = tf.random.uniform((1, 32, 32, 3))   # 1 query image

gallery_emb = tf.math.l2_normalize(cnn(gallery_imgs), axis=-1)  # (8, 32)
query_emb   = tf.math.l2_normalize(cnn(query_img),    axis=-1)  # (1, 32)

scores   = tf.matmul(query_emb, gallery_emb, transpose_b=True)[0]  # (8,)
top_cnn  = tf.argsort(scores, direction='DESCENDING').numpy()[:3]
print("[CNN]  top-3 gallery indices:", top_cnn,
      " scores:", scores.numpy()[top_cnn].round(4))

# ----- Approach B: ORB BFMatcher ----------------------------------------
orb = cv2.ORB_create(nfeatures=100)

def orb_kp_des(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    kp, des = orb.detectAndCompute(gray, None)
    return des   # may be None if no keypoints found

q_np  = (np.random.rand(32, 32, 3) * 255).astype(np.uint8)
gal_np = [(np.random.rand(32, 32, 3) * 255).astype(np.uint8) for _ in range(8)]

q_des = orb_kp_des(q_np)
bf    = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

match_counts = []
for g in gal_np:
    g_des = orb_kp_des(g)
    if q_des is not None and g_des is not None and len(q_des) and len(g_des):
        matches = bf.match(q_des, g_des)
        match_counts.append(len(matches))
    else:
        match_counts.append(0)

top_orb = np.argsort(match_counts)[::-1][:3]
print("[ORB]  top-3 gallery indices:", top_orb,
      " match counts:", [match_counts[i] for i in top_orb])

[CNN]  top-3 gallery indices: [7 6 1]  scores: [1. 1. 1.]
[ORB]  top-3 gallery indices: [7 6 5]  match counts: [0, 0, 0]


### E7) Text to Image (Conditional Generation)

**Strategy**: Conditional VAE — encode text description to a latent vector, then decode to an image.  
**Alternatives**: GAN with text conditioning (AttnGAN) · Diffusion model with text cross-attention · VQ-VAE + transformer prior.

In [67]:
# E7) Text to Image — Conditional VAE (cVAE)
# ===========================================
# Encoder : text description → latent mean & log-variance (μ, logσ²)
# Reparameterisation: z = μ + σ * ε   (ε ~ N(0,1))
# Decoder : z → Conv2DTranspose stack → generated image
#
# Alternative: replace text encoder with a transformer for richer conditioning.

import tensorflow as tf
import numpy as np

VOCAB   = ["<PAD>","a","cat","dog","sun","moon","car","plane","red","blue"]
V       = len(VOCAB)
EMB_D   = 16
LATENT  = 32

# ----- Build lightweight text encoder blocks -----------------------------
text_emb   = tf.keras.layers.Embedding(V, EMB_D, mask_zero=True)
text_pool  = tf.keras.layers.GlobalAveragePooling1D()
text_dense = tf.keras.layers.Dense(64, activation='relu')
mean_head  = tf.keras.layers.Dense(LATENT)
logv_head  = tf.keras.layers.Dense(LATENT)

# ----- Build decoder blocks ----------------------------------------------
dec_dense = tf.keras.layers.Dense(4 * 4 * 64, activation='relu')
dec_up1   = tf.keras.layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')
dec_up2   = tf.keras.layers.Conv2DTranspose(3, 3, strides=2, padding='same', activation='sigmoid')

# ----- Forward pass ------------------------------------------------------
tokens = tf.constant([[2, 3, 0, 0, 0, 0]])  # "a cat"
text_x = text_emb(tokens)
text_x = text_pool(text_x)
text_x = text_dense(text_x)
z_mean = mean_head(text_x)
z_logv = logv_head(text_x)

# Reparameterise in eager mode
epsilon = tf.random.normal(tf.shape(z_mean))
z = z_mean + tf.exp(0.5 * z_logv) * epsilon

# Decode latent vector into a small RGB image
dec = dec_dense(z)
dec = tf.reshape(dec, (-1, 4, 4, 64))
dec = dec_up1(dec)
gen_image = dec_up2(dec)

print("Text input tokens :", tokens.numpy())
print("Latent z shape    :", z.shape)          # (1, 32)
print("Generated image   :", gen_image.shape)   # (1, 16, 16, 3)
print("Pixel range       :", float(tf.reduce_min(gen_image)), "–", float(tf.reduce_max(gen_image)))

Text input tokens : [[2 3 0 0 0 0]]
Latent z shape    : (1, 32)
Generated image   : (1, 16, 16, 3)
Pixel range       : 0.4614273011684418 – 0.5433361530303955


### E8) Text to Video

**Strategy**: Text → latent vector via LSTM encoder. Temporal decoder (LSTM per step) generates a sequence of frame latents, each decoded to an image by a Conv2DTranspose head.  
**Alternatives**: Text → 3D-CNN decoder · Text → flow field sequence · Diffusion model with temporal attention blocks.

In [68]:
# E8) Text to Video — LSTM Text Encoder + Temporal Frame Decoder
# ===============================================================
# Step 1: Encode text description to a context vector.
# Step 2: LSTM temporal decoder generates T frame latents from context.
# Step 3: Each frame latent is decoded to a pixel image via Conv2DTranspose.

import tensorflow as tf
import numpy as np

VOCAB = ["<PAD>","a","cat","runs","across","the","field","dog","jumps","over"]
V     = len(VOCAB)
FRAMES, LATENT, IMG = 4, 32, 8   # 4 frames, 32-d latent, 8×8 output image

# ----- Text encoder -------------------------------------------------------
txt_emb     = tf.keras.layers.Embedding(V, 16, mask_zero=True)
txt_lstm    = tf.keras.layers.LSTM(64, return_state=True)

def encode_text(ids):
    _, h, c = txt_lstm(txt_emb(ids))  # (B, 64) context
    return h   # use h as conditioning vector

# ----- Temporal LSTM decoder: generate FRAMES latent vectors -------------
temp_lstm   = tf.keras.layers.LSTM(64, return_sequences=True, return_state=False)

def temporal_decode(context, n_frames):
    # Repeat context n_frames times to serve as LSTM input sequence
    seq = tf.tile(tf.expand_dims(context, 1), [1, n_frames, 1])  # (B, T, 64)
    return temp_lstm(seq)                                          # (B, T, 64)

# ----- Frame decoder: latent → 8×8 RGB image (Conv2DTranspose) -----------
def decode_frame(latent_vec):
    x = tf.keras.layers.Dense(4 * 4 * 16, activation='relu')(latent_vec)
    x = tf.reshape(x, (-1, 4, 4, 16))
    x = tf.keras.layers.Conv2DTranspose(8, 3, strides=2, padding='same', activation='relu')(x)
    # Output: (B, 8, 8, 8) → project to 3 channels
    return tf.keras.layers.Conv2D(3, 1, activation='sigmoid')(x)   # (B, 8, 8, 3)

# ----- Forward pass -------------------------------------------------------
tokens   = tf.constant([[2, 3, 4, 5, 6, 0]])           # "a cat runs across the field"
ctx      = encode_text(tokens)                           # (1, 64)
frame_latents = temporal_decode(ctx, FRAMES)             # (1, 4, 64)

frames = []
for t in range(FRAMES):
    frame = decode_frame(frame_latents[:, t, :])         # (1, 8, 8, 3)
    frames.append(frame)

video = tf.stack(frames, axis=1)   # (1, T, 8, 8, 3)
print("Text  :", tokens.numpy())
print("Video shape:", video.shape)  # (1, 4, 8, 8, 3)
print("Frame pixel range:", round(float(tf.reduce_min(video)), 3),
      "–", round(float(tf.reduce_max(video)), 3))

Text  : [[2 3 4 5 6 0]]
Video shape: (1, 4, 8, 8, 3)
Frame pixel range: 0.5 – 0.5


### E9) Text Similarity

**Strategy A** — dense: embed both sentences, compute cosine similarity.  
**Strategy B** — sparse: TF-IDF-style token overlap (BM25-like) computed with tensorflow-text Unicode split.  
**Alternatives**: Bi-encoder + cross-encoder re-rank · Jaccard on subword tokens · Sentence-BERT fine-tuning with STS pairs.

In [70]:
# E9) Text Similarity — Dense Cosine + Sparse Token-Overlap (BM25-style)
# =======================================================================
# Strategy A (dense) : Embedding + GlobalAvgPool → cosine similarity.
# Strategy B (sparse): Word overlap ratio (Jaccard) — fast baseline.
# Strategy C (Unicode): Character-level overlap with TensorFlow string ops.

import tensorflow as tf
import numpy as np

try:
    import tensorflow_text as tf_text
    TF_TEXT_OK = True
except Exception:
    TF_TEXT_OK = False

# ----- sentence pairs to compare ----------------------------------------
pairs = [
    ("the cat sat on the mat", "a cat is sitting on a mat"),
    ("the cat sat on the mat", "dogs love running outside"),
    ("deep learning is powerful", "neural networks are very capable"),
]

# ----- Strategy A: dense cosine (lookup embedding + mean pool) ----------
VOCAB = list(set(" ".join([s for p in pairs for s in p]).split()))
word2id = {w: i+1 for i, w in enumerate(VOCAB)}  # 0 = PAD
V = len(word2id) + 1

emb_layer  = tf.keras.layers.Embedding(V, 32, mask_zero=True)
dense_proj = tf.keras.layers.Dense(16)

def embed_sentence(s):
    ids = [word2id.get(w, 0) for w in s.split()]
    ids = tf.constant([ids])
    mask = tf.cast(ids != 0, tf.float32)[:, :, tf.newaxis]
    v = emb_layer(ids) * mask
    pooled = tf.reduce_sum(v, 1) / (tf.reduce_sum(mask, 1) + 1e-8)
    return tf.math.l2_normalize(dense_proj(pooled), axis=-1)

# ----- Strategy B: Jaccard word overlap ---------------------------------
def jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / (len(sa | sb) + 1e-8)

# ----- Strategy C: character overlap (TensorFlow string ops) ------------
def tf_text_sim(a, b):
    chars_a = set(tf.strings.unicode_split(a, 'UTF-8').numpy().tolist())
    chars_b = set(tf.strings.unicode_split(b, 'UTF-8').numpy().tolist())
    return len(chars_a & chars_b) / (len(chars_a | chars_b) + 1e-8)

print(f"{'Pair':<6} {'Dense':>8} {'Jaccard':>10} {'Chars':>10}")
print("-" * 40)
for i, (a, b) in enumerate(pairs):
    dense  = float(tf.matmul(embed_sentence(a), embed_sentence(b), transpose_b=True)[0, 0])
    jac    = jaccard(a, b)
    chars  = tf_text_sim(a, b)
    print(f"  {i+1}    {dense:>8.4f} {jac:>10.4f} {chars:>10.4f}")

Pair      Dense    Jaccard      Chars
----------------------------------------
  1      0.4754     0.3750     0.6667
  2      0.2385     0.0000     0.3529
  3     -0.1931     0.0000     0.5238


### E10) Named Entity Recognition (NER)

**Strategy**: Embedding → Bidirectional LSTM → Dense softmax — one label per token (BIO tagging scheme).  
**Alternatives**: CNN over character n-grams for OOV robustness · CRF layer on top of BiLSTM · Transformer fine-tuning (BERT-style) · tensorflow-text WordpieceTokenizer for subword-level features.

In [71]:
# E10) Named Entity Recognition (NER) — BiLSTM + BIO Tag Classifier
# ==================================================================
# BIO scheme:
#   O  = outside any entity
#   B-PER / I-PER = begin / inside a PERSON entity
#   B-ORG / I-ORG = begin / inside an ORG entity
#   B-LOC / I-LOC = begin / inside a LOCATION entity

import tensorflow as tf
import numpy as np

# ----- vocabulary & tags -----------------------------------------------
WORD_VOCAB = ["<PAD>","<UNK>","london","apple","tim","cook","visited","the","city","inc"]
TAG_VOCAB  = ["O","B-LOC","I-LOC","B-ORG","I-ORG","B-PER","I-PER"]
W, T = len(WORD_VOCAB), len(TAG_VOCAB)
word2id = {w: i for i, w in enumerate(WORD_VOCAB)}
tag2id  = {t: i for i, t in enumerate(TAG_VOCAB)}
id2tag  = {i: t for t, i in tag2id.items()}

# ----- BiLSTM NER model -------------------------------------------------
ner_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(W, 16, mask_zero=True),
    # Bidirectional LSTM reads tokens in both directions
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True)),
    tf.keras.layers.Dense(T, activation='softmax'),  # one distribution per token
])

# ----- Synthetic training sentence: "Tim Cook visited London Apple Inc" --
# Words:  tim(4) cook(5) visited(6) london(2) apple(3) inc(9)
# Tags:   B-PER  I-PER   O         B-LOC      B-ORG    I-ORG
sentence_ids = [[4, 5, 6, 2, 3, 9]]
true_tags    = [[5, 6, 0, 1, 3, 4]]   # tag ids

x = tf.constant(sentence_ids)
y = tf.constant(true_tags)

# ----- Single forward pass + loss ---------------------------------------
logits = ner_model(x)                       # (1, 6, 7) token-level probs
pred   = tf.argmax(logits, axis=-1).numpy()[0]
loss   = tf.keras.losses.SparseCategoricalCrossentropy()(y, logits)

tokens_text = ["tim","cook","visited","london","apple","inc"]
print(f"{'Token':<10} {'True':>8} {'Pred':>8}")
print("-" * 30)
for tok, t, p in zip(tokens_text, true_tags[0], pred):
    print(f"{tok:<10} {id2tag[t]:>8} {id2tag[p]:>8}")
print(f"\nNER loss: {float(loss):.4f}")

Token          True     Pred
------------------------------
tim           B-PER    I-PER
cook          I-PER    I-PER
visited           O    B-ORG
london        B-LOC        O
apple         B-ORG    I-ORG
inc           I-ORG    I-ORG

NER loss: 1.9439


### E11) Part-of-Speech Tagging (POS)

**Strategy**: Embedding → BiLSTM → Dense softmax — one POS label per token (UD tagset subset).  
**Alternatives**: Character-level CNN features prepended to word embedding (morphology-aware) · Transformer encoder fine-tuned on UD treebank · tensorflow-text subword tokeniser with sub-token label propagation.

In [73]:
# E11) Part-of-Speech Tagging (POS) — BiLSTM + Softmax
# =====================================================
# Universal POS subset used here:
#   NOUN VERB ADJ DET ADP PRON ADV PROPN NUM PUNCT
#
# To make POS predictions accurate in practice, the model needs supervised training
# on labeled sentences. This demo includes a tiny training loop so the prediction
# quality is much better than a random forward pass.

import tensorflow as tf
import numpy as np

# ----- vocabulary & POS tags -------------------------------------------
WORD_VOCAB = ["<PAD>","<UNK>","the","quick","brown","fox","jumps","over","lazy","dog","a","cat","runs","slowly","new","york","city"]
POS_TAGS   = ["NOUN","VERB","ADJ","DET","ADP","PRON","ADV","PROPN","NUM","PUNCT"]
W, P = len(WORD_VOCAB), len(POS_TAGS)
word2id = {w: i for i, w in enumerate(WORD_VOCAB)}
pos2id  = {t: i for i, t in enumerate(POS_TAGS)}
id2pos  = {i: t for t, i in pos2id.items()}

# ----- small labeled training set --------------------------------------
# Each sequence is padded to the same length so we can train in batch.
train_sentences = [
    ["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"],
    ["a", "cat", "runs", "slowly"],
    ["new", "york", "city", "runs"],
    ["the", "dog", "runs"],
]
train_tags = [
    ["DET", "ADJ", "ADJ", "NOUN", "VERB", "ADP", "DET", "ADJ", "NOUN"],
    ["DET", "NOUN", "VERB", "ADV"],
    ["PROPN", "PROPN", "PROPN", "VERB"],
    ["DET", "NOUN", "VERB"],
]
max_len = max(len(s) for s in train_sentences)

x_train = tf.keras.preprocessing.sequence.pad_sequences(
    [[word2id.get(w, 1) for w in s] for s in train_sentences],
    maxlen=max_len,
    padding='post',
    value=0,
)
y_train = tf.keras.preprocessing.sequence.pad_sequences(
    [[pos2id[t] for t in s] for s in train_tags],
    maxlen=max_len,
    padding='post',
    value=0,
)

# ----- BiLSTM POS model -------------------------------------------------
pos_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(W, 32, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(P, activation='softmax'),
])

pos_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='token_accuracy')],
)

# Train briefly so the model can overfit this tiny demo set.
pos_model.fit(x_train, y_train, epochs=60, verbose=0)

# ----- Evaluation sentence ---------------------------------------------
sentence = ["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
true_pos  = ["DET", "ADJ", "ADJ", "NOUN", "VERB", "ADP", "DET", "ADJ", "NOUN"]
ids       = [[word2id.get(w, 1) for w in sentence]]

x = tf.constant(ids)
logits = pos_model(x)
pred   = tf.argmax(logits, axis=-1).numpy()[0]

print(f"{'Token':<8} {'True':>8} {'Pred':>8}")
print("-" * 28)
correct = 0
for tok, t, p in zip(sentence, true_pos, pred):
    match = "✓" if POS_TAGS[p] == t else "✗"
    correct += int(POS_TAGS[p] == t)
    print(f"{tok:<8} {t:>8} {POS_TAGS[p]:>8}  {match}")

accuracy = correct / len(sentence)
print(f"\nToken accuracy: {accuracy:.3f}")

Token        True     Pred
----------------------------
the           DET      DET  ✓
quick         ADJ      ADJ  ✓
brown         ADJ      ADJ  ✓
fox          NOUN     NOUN  ✓
jumps        VERB     VERB  ✓
over          ADP      ADP  ✓
the           DET      DET  ✓
lazy          ADJ      ADJ  ✓
dog          NOUN     NOUN  ✓

Token accuracy: 1.000


## F) Transformer-Based Embeddings

The cells below separate the four transformer embedding styles into standalone runnable examples.

In [74]:
# F1) Encoder Embeddings
# Purpose: build contextual embeddings from an input sequence using token + positional embeddings and encoder self-attention.
# Use case: sentence encoding, retrieval, classification, semantic search.

import tensorflow as tf

vocab_size = 100
seq_len = 6
embed_dim = 16
num_heads = 2

# Toy token batch
input_ids = tf.constant([[1, 2, 3, 4, 0, 0], [5, 6, 7, 8, 9, 0]], dtype=tf.int32)

# Token + positional embeddings
word_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)(input_ids)
positions = tf.range(start=0, limit=seq_len, delta=1)
pos_emb = tf.keras.layers.Embedding(seq_len, embed_dim)(positions)
pos_emb = tf.expand_dims(pos_emb, axis=0)
encoded_input = word_emb + pos_emb

# Self-attention creates contextual encoder embeddings
encoder_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
encoder_embeddings = encoder_attn(encoded_input, encoded_input)
encoder_pooled = tf.reduce_mean(encoder_embeddings, axis=1)

print("Encoder embeddings shape:", encoder_embeddings.shape)
print("Encoder pooled shape:", encoder_pooled.shape)

Encoder embeddings shape: (2, 6, 16)
Encoder pooled shape: (2, 16)


In [75]:
# F2) Decoder Embeddings
# Purpose: construct causal decoder embeddings that only attend to past tokens.
# Use case: autoregressive generation, translation decoder, code completion.

import tensorflow as tf

vocab_size = 120
seq_len = 6
embed_dim = 16
num_heads = 2

# Toy decoder input with <SOS> prefix and future tokens masked by causality
decoder_ids = tf.constant([[1, 10, 11, 12, 0, 0], [1, 20, 21, 22, 23, 0]], dtype=tf.int32)

token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)(decoder_ids)
positions = tf.range(start=0, limit=seq_len, delta=1)
pos_emb = tf.keras.layers.Embedding(seq_len, embed_dim)(positions)
pos_emb = tf.expand_dims(pos_emb, axis=0)
decoder_input = token_emb + pos_emb

# Causal self-attention ensures no future token leakage
causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
decoder_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
decoder_embeddings = decoder_attn(decoder_input, decoder_input, attention_mask=causal_mask)
decoder_state = tf.reduce_mean(decoder_embeddings, axis=1)

print("Decoder embeddings shape:", decoder_embeddings.shape)
print("Decoder state shape:", decoder_state.shape)

Decoder embeddings shape: (2, 6, 16)
Decoder state shape: (2, 16)


In [76]:
# F3) Encoder-Decoder Embeddings
# Purpose: combine encoder context with decoder queries through cross-attention.
# Use case: machine translation, summarization, speech-to-text, captioning.

import tensorflow as tf

src_vocab = 100
tgt_vocab = 100
seq_len = 5
embed_dim = 16
num_heads = 2

# Toy source and target tokens
source_ids = tf.constant([[1, 2, 3, 4, 0], [5, 6, 7, 0, 0]], dtype=tf.int32)
target_ids = tf.constant([[1, 9, 8, 7, 0], [1, 4, 3, 2, 1]], dtype=tf.int32)

# Encoder embeddings
src_tok = tf.keras.layers.Embedding(src_vocab, embed_dim)(source_ids)
src_pos = tf.keras.layers.Embedding(seq_len, embed_dim)(tf.range(seq_len))
src_pos = tf.expand_dims(src_pos, axis=0)
encoder_input = src_tok + src_pos
encoder_layer = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
encoder_embeddings = encoder_layer(encoder_input, encoder_input)

# Decoder embeddings
tgt_tok = tf.keras.layers.Embedding(tgt_vocab, embed_dim)(target_ids)
tgt_pos = tf.keras.layers.Embedding(seq_len, embed_dim)(tf.range(seq_len))
tgt_pos = tf.expand_dims(tgt_pos, axis=0)
decoder_input = tgt_tok + tgt_pos

# Cross-attention from decoder to encoder outputs
cross_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
encoder_decoder_embeddings = cross_attn(decoder_input, encoder_embeddings, encoder_embeddings)
combined = tf.reduce_mean(encoder_decoder_embeddings, axis=1)

print("Encoder-decoder embeddings shape:", encoder_decoder_embeddings.shape)
print("Combined sequence embedding shape:", combined.shape)

Encoder-decoder embeddings shape: (2, 5, 16)
Combined sequence embedding shape: (2, 16)


In [77]:
# F4) Attention-Based Embeddings
# Purpose: turn token sequences into embeddings using learned attention weights.
# Use case: sentence pooling, keyphrase emphasis, contextual importance weighting.

import tensorflow as tf

vocab_size = 100
seq_len = 6
embed_dim = 16
num_heads = 2

# Toy sequence where one token should receive more attention
input_ids = tf.constant([[1, 2, 3, 4, 0, 0], [5, 6, 7, 8, 9, 0]], dtype=tf.int32)

# Token embeddings with positions
x = tf.keras.layers.Embedding(vocab_size, embed_dim)(input_ids)
pos = tf.keras.layers.Embedding(seq_len, embed_dim)(tf.range(seq_len))
pos = tf.expand_dims(pos, axis=0)
x = x + pos

# Attention weights over the sequence
attention_layer = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
attention_output = attention_layer(x, x)
attention_scores = tf.keras.layers.Dense(1)(attention_output)
attention_weights = tf.nn.softmax(attention_scores, axis=1)
attention_embedding = tf.reduce_sum(attention_output * attention_weights, axis=1)

print("Attention output shape:", attention_output.shape)
print("Attention embedding shape:", attention_embedding.shape)

Attention output shape: (2, 6, 16)
Attention embedding shape: (2, 16)


## G) Additional Embedding Families

This section adds compact runnable demos for the remaining embedding families requested by architecture.

| Family | Covered subtypes |
|---|---|
| Word Prediction Architectures | CBOW, Skip-Gram, Negative Sampling, Hierarchical Softmax |
| Neural Network Based Embeddings | Feedforward, Autoencoder, Siamese, Contrastive |
| Matrix Factorization Based Embeddings | SVD, LSA, Collaborative Filtering |
| Tabular / Dataframe Embeddings | Entity, Categorical, Feature |
| Graph Embeddings | Node, Edge, Graph |
| Recurrent Neural Network Embeddings | LSTM, GRU, Seq2Seq |
| Graph Neural Network Embeddings | GCN, GraphSAGE, GAT |
| Probabilistic / Statistical Embeddings | Bayesian, Gaussian, Topic |

In [78]:
# G1) Word Prediction Architectures: CBOW, Skip-Gram, Negative Sampling, Hierarchical Softmax
# Purpose: show classic word embedding training objectives used in word2vec-style models.

import tensorflow as tf
import numpy as np

vocab_size = 12
embed_dim = 8
context_ids = tf.constant([[1, 2, 3, 4], [2, 3, 4, 5]], dtype=tf.int32)
center_ids = tf.constant([[3], [4]], dtype=tf.int32)

# Shared embedding table for word prediction tasks
word_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)
context_vec = tf.reduce_mean(word_emb(context_ids), axis=1)
center_vec = tf.squeeze(word_emb(center_ids), axis=1)

# CBOW: context -> center word logits
cbow_logits = tf.keras.layers.Dense(vocab_size)(context_vec)
cbow_pred = tf.argmax(cbow_logits, axis=-1)

# Skip-Gram: center -> context word logits (predict a context word from the center)
skipgram_logits = tf.keras.layers.Dense(vocab_size)(center_vec)
skipgram_pred = tf.argmax(skipgram_logits, axis=-1)

# Negative Sampling: positive dot products vs sampled negatives
neg_samples = tf.constant([6, 7, 8], dtype=tf.int32)
neg_vecs = word_emb(neg_samples)
pos_score = tf.reduce_sum(context_vec * tf.squeeze(word_emb(tf.constant([[3], [4]])), axis=1), axis=-1)
neg_score = tf.reduce_sum(context_vec[:, None, :] * neg_vecs[None, :, :], axis=-1)
neg_sampling_loss = tf.reduce_mean(tf.nn.softplus(-pos_score)) + tf.reduce_mean(tf.nn.softplus(neg_score))

# Hierarchical Softmax: approximate a binary-tree style probability with a small dense layer stack
hs_hidden = tf.keras.layers.Dense(8, activation='relu')(context_vec)
hs_logits = tf.keras.layers.Dense(vocab_size)(hs_hidden)
hs_probs = tf.nn.softmax(hs_logits, axis=-1)

print("CBOW predictions:", cbow_pred.numpy())
print("Skip-gram predictions:", skipgram_pred.numpy())
print("Negative sampling loss:", float(neg_sampling_loss))
print("Hierarchical softmax probs shape:", hs_probs.shape)

CBOW predictions: [9 9]
Skip-gram predictions: [2 2]
Negative sampling loss: 1.3850431442260742
Hierarchical softmax probs shape: (2, 12)


In [79]:
# G2) Neural Network Based Embeddings: Feedforward, Autoencoder, Siamese, Contrastive
# Purpose: compare learned latent vectors produced by standard neural networks.

import tensorflow as tf
import numpy as np

x = tf.random.normal((4, 10))

# Feedforward embeddings
ff_emb = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8),
])(x)

# Autoencoder embeddings
encoder = tf.keras.Sequential([
    tf.keras.layers.Dense(12, activation='relu'),
    tf.keras.layers.Dense(6, activation='relu'),
])
ae_latent = encoder(x)
ae_recon = tf.keras.Sequential([
    tf.keras.layers.Dense(12, activation='relu'),
    tf.keras.layers.Dense(10),
])(ae_latent)

# Siamese embeddings: same encoder for two inputs
siamese_encoder = tf.keras.Sequential([
    tf.keras.layers.Dense(12, activation='relu'),
    tf.keras.layers.Dense(8),
])
a = tf.random.normal((4, 10))
b = a + 0.05 * tf.random.normal((4, 10))
a_emb = siamese_encoder(a)
b_emb = siamese_encoder(b)
siamese_distance = tf.norm(a_emb - b_emb, axis=-1)

# Contrastive embeddings: similar pairs close, dissimilar pairs far
labels = tf.constant([1.0, 1.0, 0.0, 0.0])
distances = tf.norm(a_emb - b_emb, axis=-1)
contrastive_loss = tf.reduce_mean(labels * tf.square(distances) + (1 - labels) * tf.square(tf.nn.relu(1.0 - distances)))

print("Feedforward embedding shape:", ff_emb.shape)
print("Autoencoder latent shape:", ae_latent.shape)
print("Autoencoder reconstruction shape:", ae_recon.shape)
print("Siamese distances:", siamese_distance.numpy())
print("Contrastive loss:", float(contrastive_loss))

Feedforward embedding shape: (4, 8)
Autoencoder latent shape: (4, 6)
Autoencoder reconstruction shape: (4, 10)
Siamese distances: [0.10693151 0.16831608 0.04887087 0.06359927]
Contrastive loss: 0.45531439781188965


In [81]:
# G3) Matrix Factorization Based Embeddings: SVD, LSA, Collaborative Filtering
# Purpose: factorize matrices into lower-dimensional latent representations.

import tensorflow as tf
import numpy as np

# Toy user-item / term-document matrix
M = tf.constant([
    [5.0, 4.0, 0.0, 1.0],
    [4.0, 0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 5.0],
    [0.0, 0.0, 5.0, 4.0],
], dtype=tf.float32)

# SVD embeddings: left/right singular vectors scaled by singular values
s, u, v = tf.linalg.svd(M, full_matrices=False)
svd_user_emb = u[:, :2] * tf.sqrt(s[:2])
svd_item_emb = tf.transpose(v[:2, :]) * tf.sqrt(s[:2])

# LSA embeddings: treat each row as a document embedding after SVD projection
lsa_doc_emb = u[:, :2] * s[:2]

# Collaborative filtering embeddings: learn user/item factors via dot products
user_factors = tf.Variable(tf.random.normal((4, 2)))
item_factors = tf.Variable(tf.random.normal((4, 2)))
cf_scores = tf.matmul(user_factors, item_factors, transpose_b=True)

print("SVD user embedding shape:", svd_user_emb.shape)
print("SVD item embedding shape:", svd_item_emb.shape)
print("LSA document embedding shape:", lsa_doc_emb.shape)
print("Collaborative filtering score matrix shape:", cf_scores.shape)

SVD user embedding shape: (4, 2)
SVD item embedding shape: (4, 2)
LSA document embedding shape: (4, 2)
Collaborative filtering score matrix shape: (4, 4)


In [82]:
# G4) Tabular / Dataframe Embeddings: Entity, Categorical, Feature
# Purpose: combine learned entity lookup embeddings with categorical and numerical feature projections.

import tensorflow as tf
import numpy as np

# Entity embeddings (e.g., user_id, item_id)
user_ids = tf.constant([1, 2, 3, 4], dtype=tf.int32)
item_ids = tf.constant([10, 11, 12, 13], dtype=tf.int32)
user_entity_emb = tf.keras.layers.Embedding(100, 8)(user_ids)
item_entity_emb = tf.keras.layers.Embedding(200, 8)(item_ids)

# Categorical embeddings (e.g., country, device)
country_ids = tf.constant([1, 3, 2, 4], dtype=tf.int32)
device_ids = tf.constant([2, 1, 2, 3], dtype=tf.int32)
country_cat_emb = tf.keras.layers.Embedding(20, 4)(country_ids)
device_cat_emb = tf.keras.layers.Embedding(10, 4)(device_ids)

# Feature embeddings: numerical projection + categorical fusion
numeric_features = tf.random.normal((4, 5))
numeric_proj = tf.keras.layers.Dense(8, activation='relu')(numeric_features)
feature_embedding = tf.concat([user_entity_emb, item_entity_emb, country_cat_emb, device_cat_emb, numeric_proj], axis=-1)

print("Entity user embedding shape:", user_entity_emb.shape)
print("Entity item embedding shape:", item_entity_emb.shape)
print("Categorical country embedding shape:", country_cat_emb.shape)
print("Feature embedding shape:", feature_embedding.shape)

Entity user embedding shape: (4, 8)
Entity item embedding shape: (4, 8)
Categorical country embedding shape: (4, 4)
Feature embedding shape: (4, 32)


In [83]:
# G5) Graph Embeddings: Node, Edge, Graph
# Purpose: encode graph structure at node, edge, and whole-graph levels.

import tensorflow as tf
import numpy as np

# Tiny graph with 4 nodes and edge list
node_features = tf.random.normal((4, 6))
edge_index = tf.constant([[0, 1], [1, 2], [2, 3], [3, 0]], dtype=tf.int32)
edge_features = tf.random.normal((4, 4))

# Node embeddings: aggregate neighbor messages
adj = tf.constant([
    [1.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0, 1.0],
    [1.0, 0.0, 1.0, 1.0],
], dtype=tf.float32)
node_messages = tf.matmul(adj, node_features)
node_embeddings = tf.keras.layers.Dense(8, activation='relu')(node_messages)

# Edge embeddings: concatenate endpoint node embeddings with edge features
edge_src = tf.gather(node_embeddings, edge_index[:, 0])
edge_dst = tf.gather(node_embeddings, edge_index[:, 1])
edge_embeddings = tf.keras.layers.Dense(8, activation='relu')(
    tf.concat([edge_src, edge_dst, edge_features], axis=-1)
)

# Graph embedding: pool node embeddings to one graph vector
graph_embedding = tf.reduce_mean(node_embeddings, axis=0, keepdims=True)

print("Node embeddings shape:", node_embeddings.shape)
print("Edge embeddings shape:", edge_embeddings.shape)
print("Graph embedding shape:", graph_embedding.shape)

Node embeddings shape: (4, 8)
Edge embeddings shape: (4, 8)
Graph embedding shape: (1, 8)


In [84]:
# G6) Recurrent Neural Network Embeddings: LSTM, GRU, Seq2Seq
# Purpose: encode order-sensitive sequences using recurrent units and seq2seq context vectors.

import tensorflow as tf
import numpy as np

seq = tf.random.normal((3, 7, 10))

# LSTM embeddings
lstm_emb = tf.keras.layers.LSTM(12)(seq)

# GRU embeddings
gru_emb = tf.keras.layers.GRU(12)(seq)

# Seq2Seq embeddings: encoder state becomes sequence summary
enc_lstm = tf.keras.layers.LSTM(12, return_state=True)
_, h, c = enc_lstm(seq)
seq2seq_emb = tf.concat([h, c], axis=-1)

print("LSTM embedding shape:", lstm_emb.shape)
print("GRU embedding shape:", gru_emb.shape)
print("Seq2Seq embedding shape:", seq2seq_emb.shape)

LSTM embedding shape: (3, 12)
GRU embedding shape: (3, 12)
Seq2Seq embedding shape: (3, 24)


In [85]:
# G7) Graph Neural Network Embeddings: GCN, GraphSAGE, GAT
# Purpose: apply message passing with different neighborhood aggregation styles.

import tensorflow as tf
import numpy as np

node_x = tf.random.normal((4, 6))
adj = tf.constant([
    [1.0, 1.0, 0.0, 0.0],
    [1.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0, 1.0],
    [0.0, 0.0, 1.0, 1.0],
], dtype=tf.float32)

# GCN embeddings: normalized neighborhood aggregation
rowsum = tf.reduce_sum(adj, axis=-1, keepdims=True)
gcn_agg = tf.matmul(adj / rowsum, node_x)
gcn_emb = tf.keras.layers.Dense(8, activation='relu')(gcn_agg)

# GraphSAGE embeddings: concat self + neighbor summary
neigh_mean = tf.matmul(adj / rowsum, node_x)
graphsage_emb = tf.keras.layers.Dense(8, activation='relu')(tf.concat([node_x, neigh_mean], axis=-1))

# GAT embeddings: attention-style weighted neighborhood aggregation
attn_scores = tf.keras.layers.Dense(1)(node_x)
attn_weights = tf.nn.softmax(attn_scores, axis=0)
gat_context = tf.reduce_sum(node_x * attn_weights, axis=0, keepdims=True)
gat_emb = tf.keras.layers.Dense(8, activation='relu')(tf.repeat(gat_context, repeats=4, axis=0))

print("GCN embedding shape:", gcn_emb.shape)
print("GraphSAGE embedding shape:", graphsage_emb.shape)
print("GAT embedding shape:", gat_emb.shape)

GCN embedding shape: (4, 8)
GraphSAGE embedding shape: (4, 8)
GAT embedding shape: (4, 8)


In [86]:
# G8) Probabilistic / Statistical Embeddings: Bayesian, Gaussian, Topic
# Purpose: represent uncertainty and latent latent structure with distributions.

import tensorflow as tf
import numpy as np

x = tf.random.normal((4, 10))

# Bayesian embeddings: mean + uncertainty (e.g., variational posterior parameters)
bayes_mean = tf.keras.layers.Dense(8)(x)
bayes_logvar = tf.keras.layers.Dense(8)(x)
bayes_sample = bayes_mean + tf.exp(0.5 * bayes_logvar) * tf.random.normal(tf.shape(bayes_mean))

# Gaussian embeddings: explicit Gaussian latent vector parameterization
gauss_mean = tf.keras.layers.Dense(8)(x)
gauss_sigma = tf.nn.softplus(tf.keras.layers.Dense(8)(x)) + 1e-6
gauss_sample = gauss_mean + gauss_sigma * tf.random.normal(tf.shape(gauss_mean))

# Topic embeddings: topic mixture over latent topics
topic_logits = tf.keras.layers.Dense(5)(x)
topic_probs = tf.nn.softmax(topic_logits, axis=-1)
topic_emb = tf.keras.layers.Dense(8)(topic_probs)

print("Bayesian mean shape:", bayes_mean.shape)
print("Bayesian sample shape:", bayes_sample.shape)
print("Gaussian sample shape:", gauss_sample.shape)
print("Topic embedding shape:", topic_emb.shape)

Bayesian mean shape: (4, 8)
Bayesian sample shape: (4, 8)
Gaussian sample shape: (4, 8)
Topic embedding shape: (4, 8)


## Edge Conditions Checklist

Use this checklist when adapting any embedding example from this notebook:
- Empty input: no tokens, no pixels, no graph nodes, no frames
- Short input: sequences shorter than the model window or patch size
- Long input: truncation, chunking, or sliding windows
- OOV input: unknown words, new ids, unseen categories
- PAD handling: ensure masked positions do not affect pooling or loss
- Missing modality: text without image, image without text, graph without edges
- Imbalanced labels: rare tags or rare classes in NER, POS, and retrieval tasks
- Noisy input: typos, blur, compression, occlusion, missing frames
- No features found: zero ORB keypoints, empty optical flow, empty captions
- Cold start: new user, new item, new document, new graph node
- Numeric stability: divide-by-zero, empty averages, very small norms
- Training vs inference skew: preprocessing and tokenization must match

Practical rule: if an input can be empty or truncated, guard it before embedding and provide a fallback vector or skip path.

## Family Edge-Case Summary

| Family | Common edge cases | Practical guardrail |
|---|---|---|
| Word prediction | tiny vocab, repeated tokens, OOV, subword mismatch | use masking, sample negatives carefully, keep a stable tokenizer |
| Neural network based | unstable training, overfitting, latent collapse | start small, regularize, compare against a baseline |
| Matrix factorization | sparse matrices, cold start, rank mismatch | add priors or fallback features for unseen rows/cols |
| Tabular / dataframe | missing values, skewed categories, rare ids | impute, bucket rare categories, normalize numeric features |
| Graph | isolated nodes, missing edges, tiny graphs | add self-loops, validate adjacency, use fallback node features |
| Recurrent | short sequences, long sequences, padding noise | pad/mask consistently, clip gradients, limit sequence length |
| GNN | disconnected components, sparse neighborhoods, oversmoothing | normalize neighborhoods, stack fewer layers, add residuals |
| Probabilistic / statistical | empty topics, tiny variance, degenerate distributions | add epsilon stabilizers, clamp variance, guard zero counts |
| Transformer | causal leakage, attention mask errors, sequence truncation | verify masks, separate encoder/decoder logic, test long inputs |
| Image / video | blur, occlusion, missing frames, no keypoints | use fallbacks, check outputs from OpenCV, skip empty descriptors |
| Mini applications | empty document, no image matches, poor labels | add fallback outputs and confidence thresholds |

Rule of thumb: if the input can be empty, truncated, or missing a modality, add a fallback vector or a skip path before training or inference.

In [88]:
# 0.5) tensorflow-text vs Keras Preprocessing: Side-by-Side Replacement Demo
# Purpose: show how to replace Keras preprocessing with tensorflow-text and vice versa.

import tensorflow as tf

sample_texts = tf.constant([
    "TensorFlow Text makes tokenization explicit and reproducible.",
    "Keras TextVectorization is convenient for quick preprocessing.",
])

print("Input texts:")
for item in sample_texts.numpy():
    print("-", item.decode())

# ---------------------------------------------------------------------
# Keras approach: TextVectorization (easy replacement for simple tokenization)
# ---------------------------------------------------------------------
keras_vectorizer = tf.keras.layers.TextVectorization(
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=10,
)
keras_vectorizer.adapt(sample_texts)
keras_ids = keras_vectorizer(sample_texts)
print("\nKeras TextVectorization ids shape:", keras_ids.shape)
print(keras_ids.numpy())

# ---------------------------------------------------------------------
# tensorflow-text approach: explicit tokenizer + lookup table
# Use this when you need WordPiece/subword control or production parity.
# ---------------------------------------------------------------------
try:
    import tensorflow_text as tf_text
    TF_TEXT_OK = True
except Exception:
    TF_TEXT_OK = False

if TF_TEXT_OK:
    tokenizer = tf_text.WhitespaceTokenizer()
    tokenized = tokenizer.tokenize(sample_texts)
    dense_tokens = tokenized.to_tensor(default_value=b"<PAD>", shape=[None, 10])

    # Build a vocabulary from observed tokens and exclude the padding token.
    flat_tokens = tf.reshape(dense_tokens, [-1])
    uniq_tokens, _ = tf.unique(flat_tokens)
    uniq_tokens = tf.boolean_mask(
        uniq_tokens,
        tf.not_equal(uniq_tokens, tf.constant(b"<PAD>")),
    )
    vocab = tf.concat([tf.constant([b"<PAD>", b"<UNK>"]), uniq_tokens], axis=0)
    lookup = tf.lookup.StaticVocabularyTable(
        tf.lookup.KeyValueTensorInitializer(vocab, tf.range(tf.shape(vocab)[0], dtype=tf.int64)),
        num_oov_buckets=1,
    )
    tftext_ids = lookup.lookup(dense_tokens)
    print("\ntensorflow-text token ids shape:", tftext_ids.shape)
    print(tftext_ids.numpy())

    # -----------------------------------------------------------------
    # Vice versa: use Keras StringLookup with a tf-text tokenizer output.
    # -----------------------------------------------------------------
    string_lookup = tf.keras.layers.StringLookup(mask_token=None)
    string_lookup.adapt(uniq_tokens)
    keras_from_tftext = string_lookup(dense_tokens)
    print("\nKeras StringLookup on tf-text tokens shape:", keras_from_tftext.shape)
    print(keras_from_tftext.numpy())
else:
    print("\ntensorflow-text is not available in this environment.")

print("\nReplacement guide:")
print("- Use Keras TextVectorization when you want one-layer preprocessing in a model.")
print("- Use tensorflow-text when you need exact tokenizer control or subword pipelines.")
print("- Use Keras StringLookup/Embedding after tensorflow-text tokenization for the model input.")

Input texts:
- TensorFlow Text makes tokenization explicit and reproducible.
- Keras TextVectorization is convenient for quick preprocessing.

Keras TextVectorization ids shape: (2, 10)
[[ 5  4  9  2 13 15  6  0  0  0]
 [10  3 11 14 12  7  8  0  0  0]]

tensorflow-text token ids shape: (2, 10)
[[ 2  3  4  5  6  7  8  0  0  0]
 [ 9 10 11 12 13 14 15  0  0  0]]

Keras StringLookup on tf-text tokens shape: (2, 10)
[[13 12  5  1  8 10  2  0  0  0]
 [14 11  6  9  7  3  4  0  0  0]]

Replacement guide:
- Use Keras TextVectorization when you want one-layer preprocessing in a model.
- Use tensorflow-text when you need exact tokenizer control or subword pipelines.
- Use Keras StringLookup/Embedding after tensorflow-text tokenization for the model input.


In [89]:
# 0.5b) Support Matrix: mirror the prose above with runnable code and comments.
replacement_map = [
    # Simple tokenization path: Keras is the shortest route for small demos.
    ("Keras TextVectorization", "tensorflow_text.WhitespaceTokenizer + lookup table"),
    # Lookup path: Keras StringLookup can consume tokenizer output from tensorflow-text.
    ("Keras StringLookup", "tf.lookup.StaticVocabularyTable or keras.layers.StringLookup"),
    # Padding path: both stacks can produce padded dense tensors for batching.
    ("TextVectorization output_sequence_length", "tf.RaggedTensor.to_tensor(default_value=...)"),
    # Subword control path: tensorflow-text is the better fit for exact subword behavior.
    ("Keras preprocessing for subwords", "tensorflow-text WordPiece/subword tokenizers"),
]

for left_side, right_side in replacement_map:
    print(f"{left_side} -> {right_side}")

# Tiny proof that the guide is code-backed: the sample text path produced token ids above.
if TF_TEXT_OK:
    print("\nCode-backed support check: tf-text token ids were generated above.")
    print("Code-backed support check: Keras token ids were generated above.")

Keras TextVectorization -> tensorflow_text.WhitespaceTokenizer + lookup table
Keras StringLookup -> tf.lookup.StaticVocabularyTable or keras.layers.StringLookup
TextVectorization output_sequence_length -> tf.RaggedTensor.to_tensor(default_value=...)
Keras preprocessing for subwords -> tensorflow-text WordPiece/subword tokenizers

Code-backed support check: tf-text token ids were generated above.
Code-backed support check: Keras token ids were generated above.
